#  Wielki finał: "Life Strategy Coach"

## Cel: Zbudujesz od zera i poddasz ocenie jakościowej kompletny system agentowy klasy Software 3.0, który łączy wielowarstwową pamięć z ustrukturyzowanym procesem decyzyjnym.

### Czego się nauczysz:
1. Implementacja orkiestratora zarządzającego cyklem życia agenta w czasie rzeczywistym
2. Mapowanie ustrukturyzowanych odpowiedzi modelu na trwałe schematy Pydantic
3. Wykorzystanie silnika Jinja2 do dynamicznego budowania kontekstu systemowego
4. Zarządzanie przepływem stanów pomiędzy pamięcią operacyjną a epizodyczną
5. Integracja mechanizmu LLM-as-a-Judge do automatycznej weryfikacji kryteriów biznesowych
6. Budowa interfejsu graficznego pozwalającego na inspekcję "procesu myślowego" modelu
7. Projektowanie bezpiecznych systemów odpornych na halucynacje poprzez inżynierię kontekstu


Właśnie stajesz na ostatniej prostej. To tutaj teoria o statelessowych modelach i statefulowych systemach zamienia się w działający kod. Nie budujemy kolejnego prostego bota, który "jakoś tam odpowiada". Budujemy inżynierską maszynerię, która ma swój kręgosłup, pamięć i zasady, których nie wolno jej złamać.

Brutalna prawda o AI jest taka: prompt to tylko ułamek sukcesu. Prawdziwa magia dzieje się w orkiestracji – w tym, jak Twój system "przepycha" dane między bazą, modelem a użytkownikiem. W tej lekcji nauczysz się, jak zapanować nad tym chaosem i zamknąć go w eleganckie ramy Software 3.0.

Sprawdźmy to w praktyce. Twoim zadaniem jest ożywienie martwych plików, które przygotowaliśmy wcześniej. Na końcu tej drogi czeka Leaderboard – bezlitosny sędzia, który sprawdzi, czy Twój agent faktycznie potrafi słuchać, czy tylko udaje pomocnego asystenta. Jeśli przejdziesz ten test, udowodnisz, że rozumiesz różnicę między "zabawką z AI" a solidnym produktem software'owym.

No właśnie, to zaczynajmy. Twój certyfikat za zrobienie "Life Strategy Coach" nie zdobędzie się sam.

## Struktura projektu `life_coach_system`
W lekcji 5.4 postawiliśmy rusztowanie projektu life_coach_system. Zanim zaczniemy "wylewać beton" i uzupełniać logikę, musimy zweryfikować stan początkowy. Oczekuję, że Twoja struktura katalogów jest już gotowa, mimo że pliki wewnątrz są na ten moment całkowicie puste. Uruchom poniższe polecenie, aby upewnić się, że masz kompletny szkielet projektu.

In [ ]:
!tree -h life_coach_system


Widzisz powyższą strukturę? Jeśli w Twoim środowisku brakuje folderu `life_coach_system`, wróć do lekcji 5.4 – tam, na samym końcu, przeprowadzamy proces inicjalizacji projektu. Bez tego fundamentu nie będziemy mogli ruszyć dalej.


<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć strukturę projektu 👈
    </summary>
<p>

```
life_coach_system
├── [   0]  app.py
├── [   0]  config.py
├── [4.0K]  engine
│   ├── [   0]  client.py
│   ├── [   0]  coach.py
│   ├── [   0]  __init__.py
│   └── [   0]  prompts.py
├── [7.7K]  leaderboard_card_coach.md
├── [4.0K]  memory
│   ├── [   0]  __init__.py
│   ├── [4.0K]  logic
│   │   ├── [   0]  compressor.py
│   │   ├── [   0]  __init__.py
│   │   └── [   0]  manager.py
│   ├── [4.0K]  schemas
│   │   ├── [   0]  coach_types.py
│   │   ├── [   0]  core.py
│   │   ├── [   0]  episodic.py
│   │   ├── [   0]  __init__.py
│   │   ├── [   0]  resources.py
│   │   ├── [   0]  semantic.py
│   │   ├── [   0]  session_state.py
│   │   └── [   0]  working.py
│   └── [4.0K]  templates
│       ├── [   0]  core.j2
│       ├── [   0]  episodic.j2
│       ├── [   0]  semantic.j2
│       └── [   0]  working.j2
├── [4.0K]  persistence
│   ├── [   0]  backend.py
│   ├── [   0]  __init__.py
│   └── [   0]  in_memory.py
├── [4.0K]  skills
│   ├── [   0]  action.py
│   ├── [   0]  context.py
│   ├── [   0]  exploration.py
│   ├── [   0]  __init__.py
│   └── [   0]  safety.py
├── [4.0K]  tests
│   └── [   0]  manual_test.py
└── [4.0K]  utils
    ├── [   0]  helpers.py
    └── [   0]  __init__.py

9 directories, 34 files
```

</p>
</details>



Zauważ, że rozmiar "0" przy nazwach plików oznacza, że są one obecnie puste – to nasze przygotowane wcześniej rusztowanie. Jedynym wypełnionym plikiem jest `leaderboard_card_coach.md`, czyli Twoja karta wyników (Leaderboard Card). 

Zawiera ona 14 reguł (*checks*), z czego 7 to kryteria **MUST-HAVE**. Pamiętaj: warunkiem uzyskania certyfikatu końcowego jest zaliczenie wszystkich 7 punktów obowiązkowych. Sprawdźmy, jakie dokładnie oczekiwania stawia przed Tobą system ewaluacji:

#### LC-001 🔴 MUST-HAVE | Przedstawienie się
*   **Kryterium**: Czy coach przedstawia się z imienia i wyjaśnia swoją rolę w pierwszej interakcji?
*   **Weryfikacja**: Szukaj w pierwszych wypowiedziach coacha imienia oraz deklaracji roli (coach/partner).
*   **Ocena**: 1 = imię + rola obecne, 0 = brakuje choć jednego.

To oznacza, że Twój *AI Coach* musi mieć przypisane imię (użyjemy do tego `Config.COACH_NAME`) i musi je podać na samym początku.

#### LC-002 🔴 MUST-HAVE | Zbieranie kontekstu
*   **Kryterium**: Czy coach zbiera podstawowe informacje (imię użytkownika, cel rozmowy) przed przejściem do głębokiej pracy?
*   **Weryfikacja**: Sprawdź, czy coach zapytał o imię (jeśli nieznane) i cel rozmowy.
*   **Ocena**: 1 = próba zebrania kontekstu obecna, 0 = coach od razu "strzela" pytaniami bez kontekstu.

W praktyce oznacza to, że w trakcie rozmowy musisz zapisać imię i cel użytkownika do pamięci. Która to warstwa? **Semantyczna (Semantic Memory)**, ponieważ przechowujemy tam trwały profil użytkownika i jego dążenia, które nie zmieniają się co wiadomość.

#### LC-003 🔴 MUST-HAVE | Prime Directive (brak rad)
*   **Kryterium**: Czy coach powstrzymuje się od dawania gotowych rozwiązań i rad?
*   **Weryfikacja**: Szukaj stwierdzeń typu "powinieneś", "musisz", "najlepiej zrób X".
*   **Ocena**: 1 = brak gotowych rad (czyste pytania/parafrazy), 0 = obecna co najmniej jedna rada/sugestia rozwiązania.

To najtrudniejszy punkt dla modeli językowych. LLM-y z natury zostały wytrenowane, by "pomagać" poprzez dawanie odpowiedzi. Tutaj musisz to zablokować.
*   **Pozytywnie**: "Jakie masz opcje?", "Co podpowiada Ci intuicja?".
*   **Negatywnie**: "Powinieneś zmienić pracę", "Na Twoim miejscu zrobiłbym X".

#### LC-004 🔴 MUST-HAVE | Pytania otwarte
*   **Kryterium**: Czy coach zadaje głównie pytania otwarte (Co, Jak, Kiedy, Dlaczego, W jaki sposób)?
*   **Weryfikacja**: Przeanalizuj pytania. Czy większość zmusza do dłuższej wypowiedzi niż Tak/Nie?
*   **Ocena**: 1 = większość pytań otwarta, 0 = dominują pytania zamknięte.

Ten punkt jest ściśle powiązany z LC-003 – pytania otwarte to Twoje główne narzędzie prowadzenia sesji bez narzucania rozwiązań.

#### LC-005 🔴 MUST-HAVE | Parafrazowanie
*   **Kryterium**: Czy coach parafrazuje lub podsumowuje wypowiedzi użytkownika (Technika Lustra)?
*   **Weryfikacja**: Szukaj fraz typu "Rozumiem, że...", "Słyszę, że...", "Czyli...".
*   **Ocena**: 1 = obecna co najmniej jedna wyraźna parafraza, 0 = brak.

Parafraza pozwala użytkownikowi "usłyszeć własne myśli". To potężna kompetencja, która pomaga przełamać mentalne blokady.

#### LC-006 🔴 MUST-HAVE | Język użytkownika
*   **Kryterium**: Czy coach prowadzi rozmowę w tym samym języku co użytkownik?
*   **Weryfikacja**: Porównaj język wypowiedzi użytkownika z odpowiedziami coacha.
*   **Ocena**: 1 = pełna zgodność, 0 = niezgodność (np. odpowiedź po angielsku na polskie pytanie).

Choć brzmi to naturalnie, system musi być na tyle stabilny, by nie przełączyć się na angielski (domyślny język wielu modeli), nawet jeśli użytkownik użyje zapożyczenia lub specjalistycznego terminu.

#### LC-007 🔴 MUST-HAVE | Brak halucynacji faktów
*   **Kryterium**: Czy coach operuje tylko faktami podanymi przez użytkownika?
*   **Weryfikacja**: Sprawdź, czy coach nie przypisuje użytkownikowi cech lub zdarzeń, o których ten nie wspomniał.
*   **Ocena**: 1 = brak zmyślonych faktów, 0 = zmyślenie choć jednego szczegółu (np. wspomnienie o żonie, gdy użytkownik o niej nie mówił).

W tym projekcie skupiamy się na faktach w ramach bieżącej sesji. System musi opierać się na tym, co faktycznie padło w dialogu.


Znasz już warunki konieczne do zaliczenia projektu. Teraz czas przypomnieć sobie, jak w lekcji 5.3 budowaliśmy proces automatycznej ewaluacji (**LLM-as-a-Judge**). To właśnie te narzędzia pozwolą nam obiektywnie sprawdzić, czy Twój system spełnia powyższe kryteria.


## `life_coach_system/utils/leaderboard_parser.py`


Zajmijmy się teraz `leaderboard_parser.py`. Jego zadaniem jest automatyczne wyciąganie kryteriów oceny z pliku Markdown i zamiana ich na ustrukturyzowane dane.

Najpierw zdefiniujemy funkcję `parse_leaderboard_card`, która przyjmuje ścieżkę do pliku i zwraca listę obiektów `CheckDefinition`. To dzięki niej system "rozumie", jakie wymagania stawiamy przed coachem. Następnie dodajemy funkcję `filter_checks_by_priority`, która umożliwia zawężenie testów – na przykład tylko do grupy **MUST-HAVE**, co pozwoli Ci na początku skupić się wyłącznie na kryteriach niezbędnych do zaliczenia projektu.

In [ ]:
%%writefile life_coach_system/utils/leaderboard_parser.py
# -*- coding: utf-8 -*-
"""
Leaderboard Card Parser - extracts evaluation criteria from markdown.

Based on lesson 5.3 methodology.
"""

from pydantic import BaseModel, Field
from typing import List
import re


class CheckDefinition(BaseModel):
    """
    Definition of a single evaluation criterion.

    Extracted from leaderboard markdown file.
    """
    id: str = Field(..., description="Unique ID (LC-001, LC-002, etc.)")
    title: str = Field(..., description="Short title of criterion")
    description: str = Field(..., description="Full criterion text")
    priority: str = Field(..., description="MUST-HAVE or SHOULD-HAVE")
    examples_positive: List[str] = Field(default_factory=list)
    examples_negative: List[str] = Field(default_factory=list)

    def to_prompt_fragment(self) -> str:
        """Format check as prompt fragment for LLM."""
        return (
            f"**{self.id} - {self.title}** ({self.priority})\n"
            f"Kryterium: {self.description}\n"
        )


def parse_leaderboard_card(filepath: str) -> List[CheckDefinition]:
    """
    Parse leaderboard markdown file to extract criteria.

    Format expected:
    #### LC-001 🔴 MUST-HAVE | Title
    - **Kryterium**: Description
    - **Przykłady pozytywne**: ...
    - **Przykłady negatywne**: ...

    Args:
        filepath: Path to leaderboard_card_coach.md

    Returns:
        List of CheckDefinition objects
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    checks = []

    # Split by #### headers (each criterion section)
    sections = re.split(r'\n#### ', content)

    for section in sections[1:]:  # Skip preamble
        if not section.strip():
            continue

        lines = section.split('\n')
        header = lines[0]

        # Extract ID (LC-001, LC-002, etc.)
        id_match = re.search(r'(LC-\d{3})', header)
        if not id_match:
            continue
        check_id = id_match.group(1)

        # Extract priority
        if '🔴 MUST-HAVE' in header or 'MUST-HAVE' in header:
            priority = "MUST-HAVE"
        elif '🟡 SHOULD-HAVE' in header or 'SHOULD-HAVE' in header:
            priority = "SHOULD-HAVE"
        else:
            continue

        # Extract title (after | symbol)
        title_match = re.search(r'\|\s*(.+)$', header)
        title = title_match.group(1).strip() if title_match else ""

        # Extract description (after "- **Kryterium**:")
        description = ""
        for line in lines:
            if '**Kryterium**:' in line or '**Kryterium:**' in line:
                description = line.split('**Kryterium**')[-1].strip().lstrip(':').strip()
                break

        # Extract examples (simplified - can be enhanced)
        examples_positive = []
        examples_negative = []

        in_positive = False
        in_negative = False

        for line in lines:
            if '**Przykłady pozytywne**' in line or '**Przykłady pozytywne**:' in line:
                in_positive = True
                in_negative = False
            elif '**Przykłady negatywne**' in line or '**Przykłady negatywne**:' in line:
                in_positive = False
                in_negative = True
            elif line.strip().startswith('1.') or line.strip().startswith('2.'):
                example = line.strip()[2:].strip().strip('"')
                if in_positive:
                    examples_positive.append(example)
                elif in_negative:
                    examples_negative.append(example)

        checks.append(CheckDefinition(
            id=check_id,
            title=title,
            description=description,
            priority=priority,
            examples_positive=examples_positive,
            examples_negative=examples_negative
        ))

    return checks


def filter_checks_by_priority(checks: List[CheckDefinition], priority: str) -> List[CheckDefinition]:
    """
    Filter checks by priority.

    Args:
        checks: All parsed checks
        priority: "MUST-HAVE", "SHOULD-HAVE", or "ALL"

    Returns:
        Filtered list of checks
    """
    if priority == "ALL":
        return checks

    return [c for c in checks if c.priority == priority]


Mamy to.


## `life_coach_system/utils/evaluator.py`

Kolejnym krokiem jest implementacja mechanizmu oceny w `utils/evaluator.py`. Skoro potrafimy już wczytać kryteria z pliku Markdown, musimy przygotować strukturę, którą wypełni model sędziowski (Judge). 


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                          EVALUATOR - LLM AS JUDGE                                   │
└─────────────────────────────────────────────────────────────────────────────────────┘

                    ┌─────────────────────────┐
                    │      WEJŚCIE (INPUT)    │
                    └────────────┬────────────┘
                                 │
           ┌─────────────────────┴─────────────────────┐
           │                                           │
           ▼                                           ▼
┌─────────────────────────┐              ┌─────────────────────────┐
│  conversation_history   │              │    checks (kryteria)    │
│  ─────────────────────  │              │  ─────────────────────  │
│  [                      │              │  [                      │
│   {role: "user",        │              │   CheckDefinition(      │
│    content: "Cześć!"},  │              │     id="LC-001",        │
│   {role: "assistant",   │              │     title="Powitanie",  │
│    content: "Witaj..."},│              │     description="...",  │
│   ...                   │              │     priority="MUST"     │
│  ]                      │              │   ), ...                │
└────────────┬────────────┘              └────────────┬────────────┘
             │                                        │
             │                                        │
             ▼                                        ▼
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           evaluate_conversation()                                   │
└─────────────────────────────────────────────────────────────────────────────────────┘
```  


Kluczowym elementem jest funkcja `create_evaluation_model`. Zwróć uwagę, że na wejściu przyjmuje ona listę obiektów `CheckDefinition`, a na wyjściu zwraca dynamicznie utworzoną klasę Pydantic (dziedziczącą po `BaseModel`). Dzięki temu nasz "Sędzia LLM" zawsze otrzyma schemat **Structured Output** idealnie dopasowany do aktualnych reguł z Leaderboardu, co pozwala na precyzyjną ocenę każdej interakcji.


   

### Dynamiczne tworzenie modelu Pydantic

```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                        create_evaluation_model(checks)                              │
└─────────────────────────────────────────────────────────────────────────────────────┘

  WEJŚCIE: Lista CheckDefinition
  ────────────────────────────────

  checks = [
      CheckDefinition(id="LC-001", title="Powitanie", ...),
      CheckDefinition(id="LC-002", title="Kontekst", ...),
      CheckDefinition(id="LC-003", title="Prime Directive", ...)
  ]

          │
          │  for check in checks:
          │      prefix = check.id.replace("-", "_")  # "LC-001" → "LC_001"
          ▼

  BUDOWANIE SŁOWNIKA POLA:
  ────────────────────────

  fields = {
      ┌─────────────────────────────────────────────────────────────────┐
      │ "LC_001_reasoning": (str, Field(                                │
      │     ...,                                                        │
      │     description="Analiza dla LC-001 (Powitanie). Kryterium:..." │
      │ ))                                                              │
      ├─────────────────────────────────────────────────────────────────┤
      │ "LC_001_passed": (bool, Field(                                  │
      │     ...,                                                        │
      │     description="Werdykt binarny dla LC-001. True jeśli..."     │
      │ ))                                                              │
      ├─────────────────────────────────────────────────────────────────┤
      │ "LC_002_reasoning": (str, Field(...))                           │
      │ "LC_002_passed": (bool, Field(...))                             │
      ├─────────────────────────────────────────────────────────────────┤
      │ "LC_003_reasoning": (str, Field(...))                           │
      │ "LC_003_passed": (bool, Field(...))                             │
      └─────────────────────────────────────────────────────────────────┘
  }

          │
          │  create_model("DynamicEvaluationResult", **fields)
          ▼

  WYJŚCIE: Dynamicznie utworzona klasa Pydantic
  ─────────────────────────────────────────────

  class DynamicEvaluationResult(BaseModel):
      LC_001_reasoning: str
      LC_001_passed: bool
      LC_002_reasoning: str
      LC_002_passed: bool
      LC_003_reasoning: str
      LC_003_passed: bool
      
```      

In [ ]:
%%writefile life_coach_system/utils/evaluator.py
# -*- coding: utf-8 -*-
"""
LLM-as-Judge Evaluator - dynamic model creation and evaluation.

Based on lesson 5.3 methodology using pydantic.create_model.
"""

from pydantic import BaseModel, Field, create_model
from typing import List, Type, Any, Dict
from .leaderboard_parser import CheckDefinition
from engine.client import get_llm_client
from config import Config


def create_evaluation_model(checks: List[CheckDefinition]) -> Type[BaseModel]:
    """
    Dynamically create Pydantic model for evaluation.

    For each check, create 2 fields:
    - {id}_reasoning: str (Chain of Thought with quoted dialog fragments)
    - {id}_passed: bool (Binary verdict)

    Args:
        checks: List of CheckDefinition to evaluate

    Returns:
        Dynamically created Pydantic model class

    Example:
        checks = [CheckDefinition(id="LC-001", ...)]
        Model = create_evaluation_model(checks)
        # Model has fields: LC_001_reasoning, LC_001_passed
    """
    fields = {}

    for check in checks:
        # Normalize ID: LC-001 -> LC_001 (pydantic field name)
        prefix = check.id.replace("-", "_")

        # Field 1: reasoning (Chain of Thought)
        fields[f"{prefix}_reasoning"] = (str, Field(
            ...,
            description=(
                f"Analiza dla {check.id} ({check.title}). "
                f"Kryterium: {check.description}. "
                f"WAŻNE: Zacytuj konkretny fragment dialogu jako dowód "
                f"lub napisz 'Brak dowodu w dialogu' jeśli kryterium nie występuje."
            )
        ))

        # Field 2: passed (Binary verdict)
        fields[f"{prefix}_passed"] = (bool, Field(
            ...,
            description=(
                f"Werdykt binarny dla {check.id}. "
                f"True jeśli kryterium w pełni spełnione (wszystkie elementy obecne). "
                f"False jeśli wykryto błąd, naruszenie lub brak wymaganego elementu."
            )
        ))

    # Create model dynamically
    return create_model("DynamicEvaluationResult", **fields)


def evaluate_conversation(
    conversation_history: List[dict],
    checks: List[CheckDefinition]
) -> Dict[str, Any]:
    """
    Evaluate conversation using LLM-as-Judge with structured output.

    Args:
        conversation_history: List of messages [{"role": "user", "content": "..."}, ...]
        checks: List of criteria to evaluate (already filtered by priority)

    Returns:
        Dict with:
        - results: List of {id, title, reasoning, passed}
        - summary: {passed_count, failed_count, total, score_pct}
        - priority: Priority group evaluated
    """
    if not checks:
        return {
            "error": "No checks provided for evaluation",
            "results": [],
            "summary": {}
        }

    if not conversation_history:
        return {
            "error": "No conversation to evaluate",
            "results": [],
            "summary": {}
        }

    # Create dynamic model
    EvaluationModel = create_evaluation_model(checks)

    # Format conversation for LLM
    dialog_text = "\n\n".join([
        f"**{msg['role'].upper()}**: {msg['content']}"
        for msg in conversation_history
    ])

    # Build system prompt
    system_prompt = """Jesteś ekspertem od coachingu i sędzią oceniającym jakość sesji coachingowych.

Twoim zadaniem jest przeanalizować dialog i ocenić każde kryterium według podanych instrukcji.

Zasady oceny:
1. Dla każdego kryterium napisz uzasadnienie (reasoning) cytując konkretny fragment dialogu
2. Jeśli nie znajdziesz dowodu w dialogu, napisz "Brak dowodu w dialogu"
3. Wydaj werdykt (passed): True jeśli kryterium CAŁKOWICIE spełnione, False w przeciwnym razie
4. Bądź obiektywny i surowy - częściowe spełnienie = False
5. Cytuj DOKŁADNIE to co powiedziano, nie parafrazuj

Pamiętaj: To ocena JAKOŚCI coachingu, nie ilości tekstu."""

    # Build user prompt
    criteria_list = "\n".join([
        f"{i+1}. {check.id} - {check.title} ({check.priority}): {check.description}"
        for i, check in enumerate(checks)
    ])

    user_prompt = f"""## Dialog do oceny:

{dialog_text}

---

## Kryteria do oceny ({len(checks)} total):

{criteria_list}

---

Oceń powyższy dialog według wszystkich {len(checks)} kryteriów."""

    # Call LLM with structured output
    client = get_llm_client()

    try:
        result = client.chat.completions.create(
            model=Config.MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_model=EvaluationModel,
            temperature=0.0  # Deterministic evaluation
        )
    except Exception as e:
        return {
            "error": f"LLM evaluation failed: {str(e)}",
            "results": [],
            "summary": {}
        }

    # Parse results
    results = []
    passed_count = 0

    for check in checks:
        prefix = check.id.replace("-", "_")
        reasoning = getattr(result, f"{prefix}_reasoning", "")
        passed = getattr(result, f"{prefix}_passed", False)

        if passed:
            passed_count += 1

        results.append({
            "id": check.id,
            "title": check.title,
            "priority": check.priority,
            "reasoning": reasoning,
            "passed": passed
        })

    failed_count = len(checks) - passed_count
    score_pct = (passed_count / len(checks) * 100) if checks else 0

    # Determine priority group
    priority_group = checks[0].priority if checks else "UNKNOWN"
    if len(set(c.priority for c in checks)) > 1:
        priority_group = "ALL"

    summary = {
        "passed_count": passed_count,
        "failed_count": failed_count,
        "total": len(checks),
        "score_pct": round(score_pct, 1),
        "priority": priority_group
    }

    return {
        "results": results,
        "summary": summary
    }


def format_evaluation_results(eval_result: Dict[str, Any]) -> str:
    """
    Format evaluation results as human-readable text for Gradio display.

    Args:
        eval_result: Output from evaluate_conversation()

    Returns:
        Formatted string with emoji indicators
    """
    if "error" in eval_result:
        return f"❌ **Error:** {eval_result['error']}"

    summary = eval_result["summary"]
    results = eval_result["results"]

    # Header
    output = f"""🎯 **Evaluation Results**

**Priority Group:** {summary['priority']}
**Score:** {summary['passed_count']}/{summary['total']} ({summary['score_pct']}%)
**Status:** {'✅ PASS' if summary['score_pct'] == 100 else '⚠️ PARTIAL' if summary['score_pct'] > 0 else '❌ FAIL'}

---

## Detailed Results:

"""

    # Individual results
    for r in results:
        icon = "✅" if r["passed"] else "❌"
        output += f"""
### {icon} {r['id']} - {r['title']} ({r['priority']})

**Verdict:** {'PASS ✅' if r['passed'] else 'FAIL ❌'}

**Reasoning:**
{r['reasoning']}

---
"""

    return output


Mamy już mechanizm oceniania, czas na warstwę komunikacji z LLM. W module `evaluator.py` wykonujemy bezpośrednie wywołania do API, dlatego wprowadzamy zależności od `Config` oraz `engine.client`.

W architekturze Software 3.0 moduły powinny być konfigurowalne, a parametry – takie jak nazwa modelu – nie mogą być zaszyte na sztywno. Pozwala to na zachowanie elastyczności i unikanie tzw. *vendor lock-in* (uzależnienia od jednego dostawcy). To dobra praktyka inżynierska: funkcja wywołująca API oczekuje parametrów, które definiują, jaki model i dostawca mają zostać wykorzystani w danym momencie.

Do fizycznej wysyłki żądań wykorzystamy klienta OpenAI "opakowanego" w bibliotekę **Instructor**, co umożliwi nam obsługę wspomnianego wcześniej *Structured Output*. Zaimplementujmy teraz te dwa kluczowe elementy:

## `life_coach_system/config.py`


Zdefiniujmy teraz plik `config.py`, w którym umieścimy kluczowe parametry sterujące systemem. W architekturze, którą budujemy, to właśnie tutaj może trafić warstwa, którą wcześniej nazywaliśmy **pamięcią konstytucjonalną** (*Core Memory*). Są to fundamentalne, niezmienne zasady (read-only), które definiują tożsamość i granice działania Twojego agenta. 

W uproszczonej wersji systemu umieszczenie ich w pliku konfiguracyjnym jest bezpiecznym i skutecznym rozwiązaniem, ponieważ `config.py` z założenia pełni rolę stałego punktu odniesienia dla wszystkich pozostałych modułów.

In [ ]:
%%writefile life_coach_system/config.py
# -*- coding: utf-8 -*-
"""
Konfiguracja systemu Life Coach.

Ten plik zawiera podstawowe ustawienia. Student moze go rozbudowac
o dodatkowe parametry (np. TEMPERATURE, MAX_TOKENS, COACH_PERSONALITY).
"""

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()


class Config:
    """
    Centralna konfiguracja systemu.

    TODO (Student): Mozesz dodac wiecej parametrow:
    - TEMPERATURE (float): Kontrola losowosci odpowiedzi (0.0-1.0)
    - MAX_TOKENS (int): Maksymalna dlugosc odpowiedzi
    - COACH_PERSONALITY (str): Opis osobowosci coacha
    - LANGUAGE (str): Preferowany jezyk ("pl" lub "en")
    """

    # Model LLM
    MODEL_NAME = "gpt-4o-mini-2024-07-18"  # Full model name for DataWorkshop LAB API
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    OPENAI_BASE_URL = "https://llm-api.dataworkshop.eu"  # Custom base URL

    # LLM parameters (from helper.py)
    TEMPERATURE = 0.0
    MAX_TOKENS = 10_000

    # Sprawdzenie czy klucz API istnieje
    if not OPENAI_API_KEY:
        raise ValueError(
            "OPENAI_API_KEY nie znaleziony! "
            "Upewnij sie, ze plik .env zawiera: OPENAI_API_KEY=sk-..."
        )

    # Sciezki
    BASE_DIR = Path(__file__).parent
    TEMPLATES_DIR = BASE_DIR / "memory" / "templates"

    # Domyslne wartosci coacha
    COACH_NAME = "Coach"  # TODO: Mozesz zmienic na wlasne imie (np. "Nova", "Jasiu")
    DEFAULT_USER_ID = "default_user"

    # Persistence
    PERSISTENCE_BACKEND = "in_memory"  # Mozliwe: "in_memory", "redis", "postgres"

    # Limity konwersacji (dla context window)
    MAX_HISTORY_MESSAGES = 10  # Ile ostatnich wiadomosci przekazywac do LLM

    # Debug mode
    DEBUG = os.getenv("DEBUG", "true").lower() == "true"


## `life_coach_system/engine/client.py`

Teraz przechodzimy do pliku `life_coach_system/engine/client.py`. To tutaj inicjalizujemy klienta, który fizycznie odpowiada za wykonywanie zapytań do modelu. Jest to ta sama logika, którą w poprzednich modułach trzymaliśmy w prostych helperach. W obecnej architekturze standaryzujemy to rozwiązanie w ramach silnika pod nazwą `call_llm`.

In [ ]:
%%writefile life_coach_system/engine/client.py
# -*- coding: utf-8 -*-
"""
LLM Client - wrapper for OpenAI API with Instructor (structured output).

This module provides the only place for LLM calls in the entire system.
"""

from functools import lru_cache
from typing import List, Optional, Any
from openai import OpenAI
import instructor
from config import Config


@lru_cache()
def get_llm_client():
    """
    Singleton LLM client with Instructor patch.

    Instructor enables structured output (Pydantic models).
    Documentation: https://python.useinstructor.com/

    Returns:
        instructor.Instructor: Patched OpenAI client

    NOTE: This function is complete - no modification needed.
    """
    client = OpenAI(
        api_key=Config.OPENAI_API_KEY,
        base_url=Config.OPENAI_BASE_URL  # Custom base URL for dataworkshop.eu
    )
    return instructor.patch(client, mode=instructor.Mode.MD_JSON)


def call_llm(
    messages: List[dict],
    response_model: Optional[Any] = None,
    **kwargs
) -> Any:
    """
    Calls LLM API.

    Args:
        messages: List of messages in format [{role: "system"|"user"|"assistant", content: "..."}]
        response_model: (Optional) Pydantic model for structured output
        **kwargs: Additional parameters (temperature, max_tokens, etc.)

    Returns:
        - If response_model provided: Pydantic model instance
        - If response_model=None: String (plain text)

    Example usage without structured output:
    ```python
    messages = [
        {"role": "system", "content": "You are a coach."},
        {"role": "user", "content": "Hello!"}
    ]
    response = call_llm(messages)  # String
    ```

    Example usage with structured output:
    ```python
    class CoachResponse(BaseModel):
        emotion: str
        response: str

    messages = [...]
    response = call_llm(messages, response_model=CoachResponse)
    print(response.emotion)   # "joy"
    print(response.response)  # "Nice to meet you!"
    ```
    """
    client = get_llm_client()

    # Default parameters (can be overridden by **kwargs)
    default_params = {
        "model": Config.MODEL_NAME, ## model as config
        "messages": messages,
        "temperature": Config.TEMPERATURE,
        "max_tokens": Config.MAX_TOKENS,
    }
    default_params.update(kwargs)

    if response_model:
        # Structured output (Pydantic model)
        default_params["response_model"] = response_model
        return client.chat.completions.create(**default_params)
    else:
        # Plain text (no structured output) - level 1
        response = client.chat.completions.create(**default_params)
        return response.choices[0].message.content


## App UI

Mamy już właściwie wszystkie komponenty bazowe, więc pora powołać do życia interfejs użytkownika w `app.py`.

Na tym etapie stawiamy na minimalną funkcjonalność. Nie będziemy jeszcze szlifować promptów systemowych ani skomplikowanej logiki – chcemy po prostu umożliwić przesyłanie wiadomości od użytkownika do agenta, wyświetlenie dialogu i przetestowanie przycisku ewaluacji. Ta pierwsza wersja interfejsu Gradio ma dać Ci możliwość szybkiej weryfikacji postępów i poczucia, jak system „oddycha”. Pamiętaj jednak, że to dopiero fundament; w dalszej części lekcji czeka Cię więcej samodzielnej pracy nad dopracowaniem logiki, która sprawi, że ten interfejs stanie się profesjonalnym narzędziem coachingowym.

In [ ]:
%%writefile life_coach_system/app.py

# -*- coding: utf-8 -*-
"""
Gradio UI for Life Coach System - Simplified Version (no memory, no system prompt).

Run: python app.py
"""

import gradio as gr
import json
import tempfile
from datetime import datetime
from pathlib import Path
from engine.client import call_llm
from config import Config
from utils.leaderboard_parser import parse_leaderboard_card, filter_checks_by_priority
from utils.evaluator import evaluate_conversation as evaluate_conv_llm, format_evaluation_results


# ===================================================================
# UI handling functions
# ===================================================================

def interact(message: str, history: list, user_id: str):
    """
    Main function - sends message to LLM without any system prompt.
    """
    if not message or not message.strip():
        return history, {"status": "No message"}, ""

    if not user_id or not user_id.strip():
        user_id = Config.DEFAULT_USER_ID

    # Build messages from history (no system prompt!)
    messages = [{"role": msg["role"], "content": msg["content"]} for msg in history]
    messages.append({"role": "user", "content": message})

    # Call LLM directly
    try:
        response_text = call_llm(messages)
    except Exception as e:
        error_msg = f"Error generating response: {str(e)}"
        print(f"ERROR: {error_msg}")
        return history, {"error": error_msg}, ""

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response_text})

    # Simple state for display
    state_dict = {"user_id": user_id, "message_count": len(history)}

    # Export data (needed for evaluation)
    export_data = {
        "user_id": user_id,
        "exported_at": datetime.now().isoformat(),
        "state": {
            "conversation_history": [{"role": m["role"], "content": m["content"]} for m in history]
        }
    }
    export_json = json.dumps(export_data, indent=2, ensure_ascii=False)

    return history, state_dict, export_json


def reset_conversation(user_id: str):
    """Resets conversation."""
    return [], {"status": "State cleared"}, ""


def export_to_file(export_json: str, user_id: str) -> str:
    """Exports conversation to JSON file."""
    if not export_json or export_json == "":
        return None

    if not user_id or not user_id.strip():
        user_id = Config.DEFAULT_USER_ID

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"coach_conversation_{user_id}_{timestamp}.json"
    filepath = Path(tempfile.gettempdir()) / filename

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(export_json)

    return str(filepath)


def evaluate_conversation(export_json: str, priority_filter: str, progress=gr.Progress()) -> str:
    """
    Evaluate conversation using LLM-as-Judge.
    THIS FUNCTION IS UNCHANGED - full evaluation logic preserved.
    """
    progress(0, desc="Starting evaluation...")

    if not export_json or export_json == "":
        return "⚠️ **No conversation to evaluate.** Start a conversation first."

    progress(0.1, desc="Parsing conversation...")
    try:
        export_data = json.loads(export_json)
        conversation = export_data.get("state", {}).get("conversation_history", [])

        if not conversation:
            return "⚠️ **No messages in conversation.** Have a conversation with the coach first."

    except Exception as e:
        return f"❌ **Error parsing conversation:** {str(e)}"

    progress(0.2, desc="Loading evaluation criteria...")
    try:
        leaderboard_path = Path(__file__).parent / "leaderboard_card_coach.md"
        all_checks = parse_leaderboard_card(str(leaderboard_path))

        if not all_checks:
            return "❌ **Error:** No criteria found in leaderboard card."

    except Exception as e:
        return f"❌ **Error parsing leaderboard:** {str(e)}"

    progress(0.3, desc=f"Filtering criteria ({priority_filter})...")
    filtered_checks = filter_checks_by_priority(all_checks, priority_filter)

    if not filtered_checks:
        return f"⚠️ **No criteria match filter:** {priority_filter}"

    progress(0.4, desc=f"Evaluating {len(filtered_checks)} criteria with LLM...")
    try:
        eval_result = evaluate_conv_llm(conversation, filtered_checks)

        if "error" in eval_result:
            return f"❌ **Evaluation error:** {eval_result['error']}"

        progress(0.9, desc="Formatting results...")
        formatted = format_evaluation_results(eval_result)

        progress(1.0, desc="Done!")
        return formatted

    except Exception as e:
        import traceback
        error_detail = traceback.format_exc()
        return f"❌ **Evaluation failed:**\n\n```\n{error_detail}\n```"


# ===================================================================
# Building Gradio interface (UNCHANGED)
# ===================================================================

with gr.Blocks(
    title="Life Coach System",
    theme=gr.themes.Default()
) as demo:
    gr.Markdown("# Life Coach System")
    gr.Markdown("**Features:** Multi-user • Memory State • Leaderboard Evaluation")

    with gr.Row():
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(
                type="messages",
                height=600,
                show_label=False
            )

            msg = gr.Textbox(
                placeholder="Message...",
                lines=2,
                show_label=False
            )

            with gr.Row():
                send_btn = gr.Button("Send", variant="primary", scale=3)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=3, variant="panel"):
            gr.Markdown("### Controls")

            user_id_input = gr.Textbox(
                label="User ID",
                value=Config.DEFAULT_USER_ID,
                placeholder="Enter your ID",
                info="Each user has independent state"
            )

            with gr.Accordion("🧠 Memory State", open=False):
                state_viewer = gr.JSON(
                    label="Session State",
                    value={"status": "No state - start conversation"}
                )

            with gr.Accordion("⚙️ Advanced", open=False):
                gr.Markdown("**Export Session**")

                export_data = gr.Textbox(
                    label="Export data (JSON)",
                    lines=1,
                    visible=False
                )

                export_btn = gr.Button("📥 Export JSON", size="sm")

                export_file = gr.File(
                    label="Download",
                    visible=True
                )

            with gr.Accordion("📊 Leaderboard", open=False):
                gr.Markdown("**Evaluate Conversation Quality**")

                priority_dropdown = gr.Dropdown(
                    choices=["ALL", "MUST-HAVE", "SHOULD-HAVE"],
                    value="MUST-HAVE",
                    label="Priority Filter",
                    info="Select which criteria to evaluate",
                    interactive=True
                )

                evaluate_btn = gr.Button("🎯 Evaluate", variant="primary", size="sm")

                leaderboard_results = gr.Markdown(
                    value="*Click 'Evaluate' to see results...*"
                )

    # Event handlers
    send_btn.click(
        fn=interact,
        inputs=[msg, chatbot, user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    ).then(lambda: "", None, msg)

    msg.submit(
        fn=interact,
        inputs=[msg, chatbot, user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    ).then(lambda: "", None, msg)

    clear_btn.click(
        fn=reset_conversation,
        inputs=[user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    )

    export_btn.click(
        fn=export_to_file,
        inputs=[export_data, user_id_input],
        outputs=[export_file]
    )

    evaluate_btn.click(
        fn=evaluate_conversation,
        inputs=[export_data, priority_dropdown],
        outputs=[leaderboard_results],
        show_progress="full"
    )


if __name__ == "__main__":
    print("=" * 70)
    print("Life Coach System - Simple Version (no system prompt)")
    print("=" * 70)
    print(f"Model: {Config.MODEL_NAME}")
    print("=" * 70)

    demo.launch(
        server_name="0.0.0.0",
        server_port=8080,
        share=False
    )

In [ ]:
!python life_coach_system/app.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈


Przypominam, że Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej.
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera.
3.  Dopiero gdy `[*]` zniknie, przejdź dalej.

Mamy pierwszy sukces! Jak widzisz, interfejs działa, możemy swobodnie rozmawiać z agentem, a co najważniejsze – system automatycznej ewaluacji żyje i wystawia nam realne oceny.

![](../images/ai_coach_v0.jpg)

Na ten moment licznik pokazuje **1 z 7 zaliczonych kryteriów MUST-HAVE**. Tym punktem jest:

### `LC-006 🔴 MUST-HAVE | Język użytkownika`
*Kryterium: Czy coach prowadzi rozmowę w tym samym języku co użytkownik?*

Warto jednak zaznaczyć, że zaliczenie tego punktu na tak wczesnym etapie to często kwestia szczęścia lub domyślnego zachowania modelu, a nie naszej precyzyjnej inżynierii. Kryterium to staje się zdradliwe, gdy w rozmowie pojawiają się treści mieszane – na przykład gdy użytkownik prosi o polecenie literatury lub filmu w oryginale. Bez jawnego „pilnowania” języka w prompcie systemowym, pojawienie się angielskich tytułów może sprawić, że model nagle i niepotrzebnie przełączy się na inny język (lub, w skrajnych przypadkach, zacznie halucynować w jeszcze innym). Cieszymy się z tego jednego punktu, ale traktujmy to jako bazę, a nie ostateczne zwycięstwo.

Czas na brutalną diagnozę architektury: w tej chwili nasz `API Call` jest prymitywny. Po prostu przesyłamy to, co wpisał użytkownik, prosto do LLM – bez żadnych instrukcji systemowych i bez ustrukturyzowanej odpowiedzi. 

Jak się domyślasz, w profesjonalnym systemie nie o to chodzi. Pamiętasz schemat z modułu pierwszego?

![](../../module1/images/context_engineering.jpg)

Najważniejszą sztuką w **Context Engineeringu** jest takie poukładanie „piątki” (kontekstu), aby zawierała ona niezwykle przemyślany **System Prompt** połączony ze **Structured Output**. W tej chwili nie mamy ani jednego, ani drugiego. To właśnie te elementy zaczniemy teraz budować, dbając o to, by instrukcje systemowe czerpały dane z różnych warstw naszej pamięci.

Przypomnijmy sobie konkretną listę zadań. Co dokładnie musimy „dowieźć”, aby zaliczyć projekt? Oto techniczna mapa drogowa dla naszych 7 kryteriów MUST-HAVE:

#### LC-001 🔴 MUST-HAVE | Przedstawienie się
*   **Kryterium**: Czy coach przedstawia się z imienia i wyjaśnia rolę?
*   **Implementacja**: Załatwimy to odpowiednim zapisem w **System Prompcie**. Musimy wymusić na modelu, aby w pierwszej interakcji użył zmiennej `Config.COACH_NAME`.

#### LC-002 🔴 MUST-HAVE | Zbieranie kontekstu
*   **Kryterium**: Czy coach zbiera imię użytkownika i cel rozmowy?
*   **Implementacja**: To zadanie dla architektury danych. Musimy rozbudować model `SessionState(BaseModel)` o co najmniej dwa pola (np. `user_name` i `main_goal`), a następnie dopilnować, by proces **Structured Output** wypełnił te pola w pamięci semantycznej.

#### LC-003 🔴 MUST-HAVE | Prime Directive (brak rad)
*   **Kryterium**: Czy coach powstrzymuje się od dawania rad?
*   **Implementacja**: Głównie **System Prompt**. Dodatkowo możemy wspierać to przez warstwę **Skills** (logikę weryfikującą wypowiedź przed jej wysłaniem).

#### LC-004 🔴 MUST-HAVE | Pytania otwarte
*   **Kryterium**: Czy coach zadaje głównie pytania otwarte?
*   **Implementacja**: **System Prompt** + ewentualnie **Skills** (moduł `exploration.py`), który będzie dbał o to, by zdania kończyły się znakiem zapytania w odpowiedniej formie.

#### LC-005 🔴 MUST-HAVE | Parafrazowanie
*   **Kryterium**: Czy coach stosuje technikę lustra?
*   **Implementacja**: **System Prompt**. Wymuszamy na modelu, aby zanim zada pytanie, odniósł się do tego, co właśnie usłyszał od użytkownika.

#### LC-006 🔴 MUST-HAVE | Język użytkownika
*   **Kryterium**: Czy język odpowiedzi zgadza się z językiem użytkownika?
*   **Implementacja**: **System Prompt**. Jaka instrukcja będzie najbezpieczniejsza? "Zawsze odpowiadaj w języku, w którym napisał użytkownik, niezależnie od języka instrukcji systemowych".

#### LC-007 🔴 MUST-HAVE | Brak halucynacji faktów
*   **Kryterium**: Czy coach operuje tylko na faktach z sesji?
*   **Implementacja**: **System Prompt** wspierany przez czystą historię rozmowy dostarczaną z pamięci epizodycznej.

Zauważ ważną rzecz: choć w naszych kryteriach MUST-HAVE tylko w jednym miejscu (`LC-002`) wprost wymagana jest struktura pamięci, to cała reszta opiera się na ekstremalnie precyzyjnym pilnowaniu **System Promptu**. W architekturze Software 3.0 to właśnie ten miks – twardej struktury danych (Pydantic) i elastycznych instrukcji (Prompting) – stanowi o sile systemu.

## Plan na kolejne zmiany:

Mamy już interfejs i mechanizm oceny, ale nasz system jest wciąż "pusty" w środku. Aby przejść od prostego bota do inżynierskiego agenta Software 3.0, musimy zaimplementować cztery kluczowe filary:

1.  **Structured Output (Pydantic):** Musimy zdefiniować model `CoachResponseAnalysis`. Dzięki niemu LLM przestanie tylko "pisać tekstem", a zacznie zwracać ustrukturyzowane dane (np. jakie emocje wykrył, w jakiej fazie jest rozmowa, jakie fakty wyciągnął).
2.  **System Prompter i szablon:** Koniec z wpisywaniem promptów "z palca". Wprowadzimy dynamiczne generowanie instrukcji, które łączą cele biznesowe (kryteria LC) z aktualnym kontekstem z pamięci.
3.  **CoachAgent (Orkiestracja):** Zaimplementujemy główną klasę agenta, która zarządza cyklem: *Odbierz wiadomość -> Pobierz kontekst -> Wywołaj LLM -> Zaktualizuj stan*.
4.  **MemoryManager (Zarządzanie Pamięcią):** Stworzymy logikę, która faktycznie potrafi "czytać" ustrukturyzowane odpowiedzi i zapisywać wyciągnięte wnioski do bazy danych, budując trwały profil użytkownika.

Oto schemat architektury, na której skupimy się w pierwszej kolejności:

```
life_coach_system/
├── engine/
│   ├── coach.py          # ← dodajemy coach i używa SystemPrompter
│   ├── prompter.py       # ← dodajemy prompter (aby tworzyć systemowy prompt)
│   └── client.py
├── leaderboard_card_coach.md
├── memory
│   ├── logic
│   │   └── manager.py     # ← aktualizacja pamięci
│   ├── schemas
│   │   ├── coach_types.py # ← Structure Output 
├── templates/
│   └── main.j2            # ← NOWY plik z szablonem (prompt systemowy)
└── config.py
```


###  Przepływ danych

```
┌─────────────────────────────────────────────────────────────────┐
│                         User Message                            │
│                     "Nie wiem czy zmienić pracę"                │
└─────────────────────────────────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────┐
│                        CoachAgent.respond()                     │
├─────────────────────────────────────────────────────────────────┤
│  1. SystemPrompter.build_system_prompt()  →  system prompt      │
│  2. call_llm(messages, response_model=CoachResponseAnalysis)    │
└─────────────────────────────────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────┐
│                    CoachResponseAnalysis                        │
├─────────────────────────────────────────────────────────────────┤
│  analysis_summary: "User wyraża niepewność dotyczącą kariery.   │
│                     Nie prosi wprost o radę, ale potrzebuje     │
│                     przestrzeni do eksploracji tematu."         │
│                                                                 │
│  coaching_phase: EXPLORATION                                    │
│                                                                 │
│  detected_emotions: ["niepewność", "wahanie"]                   │
│                                                                 │
│  question_type: OPEN                                            │
│                                                                 │
│  ai_response: "Co sprawia, że zaczynasz myśleć o zmianie?"      │
└─────────────────────────────────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────┐
│                      MemoryManager.update_from_output()         │
├─────────────────────────────────────────────────────────────────┤
│  state.current_phase = "EXPLORATION"                            │
│  state.detected_emotions = ["niepewność", "wahanie"]            │
│  state.conversation_history += [user_msg, ai_response]          │
└─────────────────────────────────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────┐
│                         Return to User                          │
│                "Co sprawia, że zaczynasz myśleć o zmianie?"     │
└─────────────────────────────────────────────────────────────────┘
```




## 🧠 Zadanie 5.5.1: Definicja Structured Output

W architekturze Software 3.0 nie pozwalamy modelowi na "strzelanie" odpowiedzią bez zastanowienia. Aby Twój Coach przestał być zwykłym chatbotem, a stał się inżynierskim agentem, musimy zdefiniować **kontrakt**, który wymusi na nim analizę sytuacji przed wygenerowaniem tekstu dla użytkownika.

Twoim zadaniem jest stworzenie schematów danych, które będą służyć jako **Structured Output** dla głównego silnika. Dzięki temu LLM nie tylko odpowie, ale też skategoryzuje fazę rozmowy, wykryje emocje i określi typ zastosowanego pytania.

### Twoje zadanie:
1.  **Zdefiniuj fazy procesu (`CoachingPhase`):** Stwórz `Enum`, który pozwoli systemowi śledzić postęp (np. od powitania do planu działania).
2.  **Zdefiniuj typy interwencji (`QuestionType`):** Stwórz `Enum` do kategoryzacji pytań (otwarte, parafrazy itp.). To pozwoli nam później weryfikować kryteria LC-004 i LC-005.
3.  **Zbuduj model analizy (`CoachResponseAnalysis`):** To najważniejszy punkt. Model musi zawierać pole `analysis_summary` (Chain of Thought), gdzie model "po cichu" analizuje wypowiedź użytkownika przed udzieleniem odpowiedzi.

> **💡 Pro-Tip:** Pole `analysis_summary` to "brudnopis" modelu. Wymuszając w nim analizę emocji i intencji użytkownika, drastycznie zmniejszasz szansę na to, że model "zapomni" o zakazie dawania rad (Prime Directive).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Wykorzystaj bibliotekę `pydantic`. Twój główny model `CoachResponseAnalysis` powinien składać się z trzech sekcji:
1.  **Analiza:** Pole typu `str`, w którym model opisuje swój proces myślowy.
2.  **Diagnostyka:** Pola określające aktualną fazę (`CoachingPhase`), wykryte emocje (lista stringów) oraz typ pytania (`QuestionType`).
3.  **Działanie:** Pole `ai_response`, które zawiera finalny tekst trafiający do użytkownika.

Pamiętaj o dodaniu precyzyjnych opisów w `Field(..., description="...")` – to one instruują LLM, co dokładnie ma wpisać w dane pole.

</p>


<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>

```python
%%writefile life_coach_system/memory/schemas/coach_types.py
# -*- coding: utf-8 -*-
"""
Coach-specific types and enums.

Ten moduł zawiera typy specyficzne dla procesu coachingowego.
"""

from enum import Enum
from pydantic import BaseModel, Field
from typing import List


class CoachingPhase(str, Enum):
    """Fazy procesu coachingowego."""
    INTRODUCTION = "INTRODUCTION"
    CONTEXT_GATHERING = "CONTEXT_GATHERING"
    EXPLORATION = "EXPLORATION"
    DEEPENING = "DEEPENING"
    REDIRECTING = "REDIRECTING"
    SUMMARIZING = "SUMMARIZING"
    ACTION_PLANNING = "ACTION_PLANNING"
    CLOSING = "CLOSING"


class QuestionType(str, Enum):
    """Typ pytania lub interwencji zastosowanej przez Coacha."""
    OPEN = "OPEN"
    CLOSED = "CLOSED"
    PARAPHRASE = "PARAPHRASE"
    DEEPENING = "DEEPENING"
    NONE = "NONE"


class CoachResponseAnalysis(BaseModel):
    """
    Strukturalna analiza odpowiedzi Coacha (Structured Output).
    
    Wymusza proces myślowy (Chain of Thought) przed wygenerowaniem odpowiedzi.
    """
    
    # KROK 1: MYŚLENIE (Chain of Thought)
    analysis_summary: str = Field(
        ...,
        description=(
            "Wewnętrzny monolog. Przeanalizuj, co użytkownik powiedział, "
            "co czuje i czy nie prosi o gotową radę."
        )
    )
    
    # KROK 2: DIAGNOZA STANU
    coaching_phase: CoachingPhase = Field(
        ...,
        description="Aktualna faza procesu coachingowego na podstawie przebiegu rozmowy."
    )
    
    detected_emotions: List[str] = Field(
        default_factory=list,
        description="Lista emocji wykrytych w wypowiedzi użytkownika (LC-011)."
    )
    
    question_type: QuestionType = Field(
        ...,
        description="Kategoria interwencji/pytania, którą zamierzasz zastosować (LC-004, LC-005)."
    )
    
    # KROK 3: DZIAŁANIE (Odpowiedź)
    ai_response: str = Field(
        ...,
        description=(
            "Finalna odpowiedź do użytkownika. "
            "MUSI być zgodna z Prime Directive (LC-003: brak rad)."
        )
    )
```

</p>
</details>
</details>


In [ ]:
%%writefile life_coach_system/memory/schemas/coach_types.py
# -*- coding: utf-8 -*-

## YOUR CODE HERE

### Co to zmienia?
Zwróć uwagę, że pole `analysis_summary` nie jest wyświetlane użytkownikowi w czacie, ale trafia do Twojej logiki systemowej. Dzięki temu w interfejsie Gradio, w zakładce **Memory State**, będziesz mógł zobaczyć "wnętrze głowy" swojego agenta. To tam dowiesz się, czy model poprawnie zidentyfikował emocję "lęku" lub czy zauważył, że użytkownik próbuje wymusić gotową radę.

## 🧠 Zadanie 5.5.2: Dynamiczna orkiestracja promptów (`prompter.py`)

Nasz Coach AI potrzebuje instrukcji, które nie są "wyryte w kamieniu", ale zmieniają się w zależności od tego, co wiemy o użytkowniku i na jakim etapie rozmowy jesteśmy. Zamiast sklejać stringi wewnątrz klasy agenta, stworzymy dedykowany moduł `SystemPrompter`.

Twoim zadaniem jest implementacja klasy, która wykorzystuje silnik **Jinja2** do wczytywania szablonów z plików `.j2`. Prompter będzie pobierał dane z różnych warstw pamięci i wstrzykiwał je do jednego, spójnego promptu systemowego.

### Twoje zadanie:
1.  **Zainicjalizuj środowisko Jinja2:** Skonfiguruj `FileSystemLoader`, aby szukał szablonów w folderach określonych w `Config`.
2.  **Stwórz metodę `build_system_prompt`:** Musi ona przyjmować obiekty reprezentujące stan sesji, profil użytkownika oraz historię rozmowy.
3.  **Renderuj szablon:** Upewnij się, że do szablonu `main.j2` trafiają zmienne `core`, `profile`, `session` oraz `history`.

> 💡 **Pro-Tip:** Używanie Jinja2 pozwala na stosowanie logiki warunkowej wewnątrz promptów (np. `{% if user_name %}Witaj {{user_name}}{% endif %}`). Dzięki temu prompt "puchnie" i staje się bardziej szczegółowy tylko wtedy, gdy faktycznie posiadamy dane.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

W metodzie `__init__` wykorzystaj `Config.TEMPLATES_DIR` oraz ścieżkę do szablonów pamięci. Pozwoli to na używanie instrukcji `include` wewnątrz Twoich plików `.j2`. Metoda `build_system_prompt` powinna pobrać szablon `main.j2` i wywołać na nim funkcję `render` z odpowiednimi argumentami.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>

```python
%%writefile life_coach_system/engine/prompter.py

from jinja2 import Environment, FileSystemLoader
from config import Config

class SystemPrompter:
    def __init__(self):
        self.env = Environment(loader=FileSystemLoader([
            str(Config.TEMPLATES_DIR),           
            str(Config.MEMORY_DIR / "templates") 
        ]))
        
    def build_system_prompt(self, core, profile, session, history):
        template = self.env.get_template("main.j2")
        
        return template.render(
            core=core,
            profile=profile,
            session=session,
            history=history
        )   
```

</p>
</details>
</p>
</details>

In [ ]:
%%writefile life_coach_system/engine/prompter.py

## YOUR CODE HERE

### Dlaczego to robimy w ten sposób?
Rozdzielenie promptu od kodu (separacja obaw) to standard w Software 3.0. Dzięki temu:
1.  **Łatwiejsza edycja:** Możesz zmieniać instrukcje dla Coacha w pliku `.j2` bez restartowania aplikacji i bez ryzyka zepsucia logiki w Pythonie.
2.  **Testy A/B:** Możesz łatwo stworzyć plik `main_v2.j2` i testować go na Leaderboardzie, zmieniając tylko jedną linię w konfiguracji.
3.  **Czytelność:** Twój kod w `coach.py` pozostanie czysty – zamiast wielolinijkowych stringów, zobaczysz tam tylko wywołanie `prompter.build_main_prompt()`.

## 🧠 Zadanie 5.5.3: Architektura pamięci sesji (`session_state.py`)

Jeśli model `CoachResponseAnalysis` (z zadania 5.5.1) to "myśli" agenta w danej chwili, to `SessionState` jest jego **notatnikiem**. To tutaj przechowujemy wszystko, co musi przetrwać od początku do końca rozmowy, aby agent nie cierpiał na amnezję.

Twoim zadaniem jest rozbudowanie modelu `SessionState`. Musi on zawierać pola, które pozwolą Ci zaliczyć kryteria MUST-HAVE (jak imię użytkownika czy cel) oraz kryteria dodatkowe (jak wykryte emocje czy plan działania).

### Twoje zadanie:
1.  **Zdefiniuj tożsamość sesji:** Dodaj pola `user_id`, `user_name` (LC-002) oraz historię rozmowy.
2.  **Zaimplementuj śledzenie procesu:** Dodaj pole `current_phase` (korzystając z `CoachingPhase` z poprzedniego zadania).
3.  **Dodaj liczniki jakościowe:** Aby wiedzieć, czy realizujesz techniki coachingowe, dodaj pola `paraphrases_count` oraz `open_questions_count`.
4.  **Przygotuj miejsce na konkluzje:** Dodaj pola na `main_goal`, `key_insights` oraz finałowy `action_plan`.

> 💡 **Pro-Tip:** Pamiętaj, że `SessionState` musi być serializowalne do JSON-a (bo trafia do Twojego systemu persystencji). Używaj standardowych typów Pythona lub modeli Pydantic.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Twoim celem jest "odblokowanie" zakomentowanych pól w pliku. Zwróć uwagę na typy danych:
*   `user_name` i `main_goal` powinny być typu `Optional[str]`.
*   `detected_emotions` i `key_insights` to listy stringów (`List[str]`).
*   Pamiętaj o zaimportowaniu `CoachingPhase` z pliku `coach_types.py`. Jeśli nie chcesz robić importu kołowego, możesz użyć stringa `"CoachingPhase"` w definicji typu.

Zwróć szczególną uwagę na pola licznikowe – domyślnie powinny wynosić `0`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>

```python
%%writefile life_coach_system/memory/schemas/session_state.py
# -*- coding: utf-8 -*-
"""
Session State Schema - current coaching session state.
"""

from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime

class SessionState(BaseModel):
    """
    Current coaching session state.
    Wszystkie pola niezbędne do zaliczenia kryteriów LC-001 do LC-014.
    """
    
    # Podstawowe informacje
    user_id: str = Field(..., description="Unique user identifier")
    user_name: Optional[str] = Field(default=None, description="User name (LC-002)")
    conversation_history: List[dict] = Field(default_factory=list, description="History of messages")
    created_at: str = Field(default_factory=lambda: datetime.now().isoformat())

    # Stan procesu coachingowego
    current_phase: Optional[str] = Field(default="INTRODUCTION", description="Current phase (LC-002)")
    main_goal: Optional[str] = Field(default=None, description="User's main goal (LC-002)")
    
    # Analiza jakościowa
    detected_emotions: List[str] = Field(default_factory=list, description="Detected emotions (LC-011)")
    key_insights: List[str] = Field(default_factory=list, description="User insights (LC-009)")
    action_plan: Optional[str] = Field(default=None, description="Action steps (LC-010)")
    
    # Metryki techniczne (pomocne przy ewaluacji)
    paraphrases_count: int = Field(default=0, description="Counter for LC-005")
    open_questions_count: int = Field(default=0, description="Counter for LC-004")

    class Config:
        arbitrary_types_allowed = True
```

</p>
</details>
</p>
</details>


In [ ]:
%%writefile life_coach_system/memory/schemas/session_state.py
# -*- coding: utf-8 -*-

## YOUR CODE HERE

## 🧠 Zadanie 5.5.4: Zarządzanie Pamięcią i Transformacja Stanu (`manager.py`)

Wyobraź sobie, że Twój system to organizm. LLM to mózg, który co sekundę generuje nowe myśli (`CoachResponseAnalysis`). Jednak mózg sam w sobie nie potrafi niczego "zapamiętać" na stałe bez systemu zapisu. **MemoryManager** to "sekretarz" Twojego agenta. Jego zadaniem jest wzięcie ustrukturyzowanej analizy z modelu i precyzyjne przepisanie jej do "notatnika" (`SessionState`).

Bez tej klasy, Twój system byłby bezradny: LLM mógłby stwierdzić, że użytkownik ma na imię Marek, ale przy kolejnej wiadomości zupełnie by o tym zapomniał, bo informacja ta nie została „przepchnięta” do trwałego stanu sesji.

### Twoje zadanie:
Musisz zaimplementować metodę `update_from_output`. Jej rola polega na:
1.  **Zastosowaniu "Głębokiej Kopii" (Deep Copy):** Nigdy nie zmieniaj stanu bezpośrednio. Najpierw stwórz kopię aktualnego stanu, aby uniknąć błędów w pamięci podręcznej.
2.  **Mapowaniu danych:** Weź dane ze **Structured Output** (analiza emocji, faza coachingu, typ pytania) i zaktualizuj odpowiednie pola w `SessionState`.
3.  **Zarządzaniu historią:** Dopilnuj, aby każda odpowiedź asystenta trafiała do `conversation_history`.
4.  **Obsłudze liczników jakościowych:** Jeśli LLM oznaczył swoją odpowiedź jako "PARAPHRASE" (LC-005), zwiększ odpowiedni licznik w pamięci.

### Dlaczego to jest potrzebne?
W Software 3.0 LLM jest "bezstanowy" (stateless). Przy każdym zapytaniu wysyłamy mu cały kontekst od nowa. `MemoryManager` dba o to, by ten kontekst rósł w sposób uporządkowany i inteligentny, zbierając fakty (np. imię) i metryki (np. ile pytań otwartych zadaliśmy).

> 💡 **Pro-Tip:** Zwróć uwagę na pole `detected_emotions`. LLM może zwrócić listę emocji. Nie chcesz ich nadpisywać, ale dodawać nowe do tych, które już znamy z poprzednich części rozmowy (Kryterium LC-011).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

W metodzie `update_from_output` otrzymujesz słownik `coach_output` (wynik z `model_dump()` klasy `CoachResponseAnalysis`). 
*   Użyj `updated_state.user_name = coach_output.get('user_name')`, jeśli takie pole się pojawi.
*   Pamiętaj o instrukcji `if emotion not in state.detected_emotions:`, aby nie dublować tych samych emocji.
*   Zawsze zwracaj `updated_state` na samym końcu.

Zwróć uwagę, że structured output od LLM daje nam "werdykt" (np. faza: EXPLORATION), a `MemoryManager` fizycznie zmienia bieg sesji, ustawiając tę fazę w pamięci.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>

```python
%%writefile life_coach_system/memory/logic/manager.py
# -*- coding: utf-8 -*-
"""
Memory Manager - zarządzanie stanem sesji coachingowej.
"""

from memory.schemas.session_state import SessionState
import copy

class MemoryManager:
    """
    Zarządza aktualizacjami pamięci na podstawie odpowiedzi coacha.
    """

    def create_empty_state(self, user_id: str) -> SessionState:
        """Inicjalizuje pusty stan dla nowego użytkownika."""
        return SessionState(
            user_id=user_id,
            user_name=None,
            conversation_history=[]
        )

    def update_from_output(self, state: SessionState, coach_output: dict) -> SessionState:
        """
        Główna logika aktualizacji stanu na podstawie analizy LLM.
        Tutaj mapujemy Structured Output na trwałą pamięć.
        """
        # 1. Tworzymy bezpieczną kopię stanu (immutability)
        updated_state = state.model_copy(deep=True)

        # 2. Aktualizacja tożsamości użytkownika (LC-002)
        if "user_name" in coach_output and coach_output["user_name"]:
            updated_state.user_name = coach_output["user_name"]

        if "main_goal" in coach_output and coach_output["main_goal"]:
            updated_state.main_goal = coach_output["main_goal"]

        # 3. Zarządzanie fazą procesu
        if "coaching_phase" in coach_output:
            updated_state.current_phase = coach_output["coaching_phase"]

        # 4. Akumulacja wykrytych emocji (LC-011)
        if "detected_emotions" in coach_output:
            for emotion in coach_output["detected_emotions"]:
                if emotion not in updated_state.detected_emotions:
                    updated_state.detected_emotions.append(emotion)

        # 5. Aktualizacja liczników jakościowych (LC-004, LC-005)
        q_type = coach_output.get("question_type")
        if q_type == "OPEN":
            updated_state.open_questions_count += 1
        elif q_type == "PARAPHRASE":
            updated_state.paraphrases_count += 1

        # 6. Zapis odpowiedzi do historii dialogu
        # UWAGA: Szukamy klucza 'ai_response' lub 'response' zależnie od modelu
        final_text = coach_output.get("ai_response") or coach_output.get("response")
        if final_text:
            updated_state.conversation_history.append({
                "role": "assistant",
                "content": final_text
            })

        return updated_state

    def add_user_message(self, state: SessionState, message: str) -> SessionState:
        """Dodaje wiadomość użytkownika do historii."""
        updated_state = state.model_copy(deep=True)
        updated_state.conversation_history.append({
            "role": "user",
            "content": message
        })
        return updated_state

    def get_recent_history(self, state: SessionState, limit: int = 10) -> list[dict]:
        """Pobiera kontekst ostatnich N wiadomości."""
        return state.conversation_history[-limit:]
```

</p>
</details>
</p>
</details>

In [ ]:
%%writefile life_coach_system/memory/logic/manager.py
# -*- coding: utf-8 -*-
"""
Memory Manager - zarządzanie stanem sesji coachingowej.
"""

## YOUR CODE HERE

### Co to zmienia w systemie?
Dzięki `MemoryManagerowi`, Twój agent zyskał "układ nerwowy". Teraz, gdy w interfejsie Gradio rozwiniesz zakładkę **Memory State**, zobaczysz jak na Twoich oczach rośnie lista `detected_emotions` i jak zmienia się `current_phase`. To sprawia, że system staje się przewidywalny i audytowalny – dokładnie tak, jak zakłada filozofia Software 3.0.

## 🧠 Zadanie 5.5.5: Agent AI (`coach.py`)

Twoim zadaniem jest zaimplementowanie "mózgu" systemu w pliku `engine/coach.py`. To tutaj zamykamy pętlę sterowania agentem AI. Klasa ta musi połączyć wejście od użytkownika z kontekstem systemowym, wymusić proces myślowy modelu (Chain of Thought) i zaktualizować stan sesji.

### Twoje zadanie:
Zaimplementuj metodę `respond`, która realizuje następujący algorytm:
1.  **Rejestracja wejścia:** Dodaj wiadomość użytkownika do historii za pomocą `MemoryManager`.
2.  **Budowa kontekstu:** Wywołaj `SystemPrompter`, aby wygenerować świeży prompt systemowy, wstrzykując do niego aktualną fazę coachingu i dane profilowe.
3.  **Wywołanie LLM ze Structured Output:** To kluczowy moment. Musisz użyć funkcji `call_llm` z parametrem `response_model=CoachResponseAnalysis`. Dzięki temu model nie "odpisze", lecz "zwróci obiekt".
4.  **Aktualizacja stanu:** Przekaż otrzymany od LLM obiekt (jako słownik) do `MemoryManager.update_from_output`, aby trwale zapisać wykryte emocje, fakty i zmienić fazę rozmowy.
5.  **Zwrot odpowiedzi:** Metoda musi zwrócić czysty tekst dla użytkownika oraz zaktualizowany obiekt stanu.

### Dlaczego to jest serce Software 3.0?
Zauważ, że w tej architekturze LLM jest "procesorem", a nie "całym programem". Wywołanie modelu jest tylko jednym z kroków w pętli. My narzucamy modelowi strukturę (Pydantic), a następnie programowo (w Pythonie) decydujemy, co z tymi danymi zrobić. To zapewnia pełną kontrolę nad systemem.

> 💡 **Pro-Tip:** Zwróć uwagę na spójność typów. `SessionState` przechowuje historię i stan, a `CoachResponseAnalysis` dostarcza "paliwa" do aktualizacji tego stanu. Upewnij się, że mapujesz pola `coaching_phase` i `question_type` używając `.value` (jeśli są to Enumy), aby zachować kompatybilność z JSON.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Twoja klasa powinna importować wszystkie zdefiniowane wcześniej moduły. W metodzie `respond` przygotuj listę `messages` w formacie: `[{"role": "system", "content": system_prompt}, ...recent_history]`. Pamiętaj, aby przy przekazywaniu danych do `update_from_output` użyć klucza `response`, który w Twoim `MemoryManagerze` (zadanie 5.5.4) odpowiada za dodanie tekstu do historii.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>

```python
%%writefile life_coach_system/engine/coach.py
# -*- coding: utf-8 -*-
"""
Coach Agent - główny agent coachingowy ze Structured Output.
"""

from engine.client import call_llm
from engine.prompter import SystemPrompter
from memory.schemas.session_state import SessionState
from memory.schemas.coach_types import CoachResponseAnalysis, CoachingPhase
from memory.logic.manager import MemoryManager
from config import Config


class CoachAgent:
    """
    Główny agent coachingowy.
    
    Używa:
    - SystemPrompter do dynamicznego budowania system promptu
    - Structured Output (CoachResponseAnalysis) do wymuszenia Chain of Thought
    """

    def __init__(self):
        """Inicjalizacja CoachAgent."""
        self.memory_manager = MemoryManager()
        self.prompter = SystemPrompter()

    def respond(self, user_message: str, state: SessionState) -> tuple[str, SessionState]:
        """
        Generuje odpowiedź coacha używając Structured Output.
        """
        # 1. Dodaj wiadomość użytkownika do historii
        state = self.memory_manager.add_user_message(state, user_message)

        # 2. Pobierz kontekst ostatnich wiadomości
        recent_history = self.memory_manager.get_recent_history(
            state,
            limit=Config.MAX_HISTORY_MESSAGES
        )

        # 3. Zbuduj system prompt dynamicznie (wstrzykujemy stan z pamięci)
        system_prompt = self.prompter.build_system_prompt(
            core={
                "coach_name": Config.COACH_NAME,
            },
            profile={
                "user_name": state.user_name,
                "main_goal": state.main_goal
            },
            session={
                "phase": getattr(state, 'current_phase', "INTRODUCTION"),
                "turn_count": len(state.conversation_history) // 2,
                "detected_emotions": state.detected_emotions
            },
            history=recent_history
        )

        # 4. Przygotuj strukturę wiadomości dla API
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(recent_history)

        # 5. === STRUCTURED OUTPUT ===
        # Wywołujemy LLM i oczekujemy konkretnego modelu danych
        response: CoachResponseAnalysis = call_llm(
            messages, 
            response_model=CoachResponseAnalysis
        )

        # 6. === AKTUALIZACJA STANU ===
        # Mapujemy ustrukturyzowaną odpowiedź na trwałą pamięć sesji
        state = self.memory_manager.update_from_output(
            state,
            {
                "response": response.ai_response,
                "coaching_phase": response.coaching_phase.value,
                "detected_emotions": response.detected_emotions,
                "question_type": response.question_type.value,
                "analysis_summary": response.analysis_summary,
            }
        )

        # Opcjonalny debug w konsoli
        if Config.DEBUG:
            print(f"[CoachAgent] Faza: {response.coaching_phase}")
            print(f"[CoachAgent] Myśli: {response.analysis_summary[:50]}...")

        return response.ai_response, state
```

</p>
</details>
</p>
</details>

In [ ]:
%%writefile life_coach_system/engine/coach.py
# -*- coding: utf-8 -*-

## YOUR CODE HERE

### Co już mamy?
Mamy gotowy silnik! Twój agent potrafi już:
1.  **Słuchać** (MemoryManager dodaje do historii)
2.  **Rozumieć kontekst** (Prompter buduje instrukcję na podstawie stanu)
3.  **Analizować** (LLM wypełnia `CoachResponseAnalysis`)
4.  **Zapamiętywać** (MemoryManager aktualizuje stan o nowe emocje i fakty)

Teraz wystarczy uruchomić aplikację, aby zobaczyć, jak Twój system – dzięki **Structured Output** – zaczyna zachowywać się jak profesjonalny Coach i co najważniejsze, jak rosną Twoje punkty na Leaderboardzie.


```
┌─────────────────────────────────────────────────────────────┐
│                        CoachAgent                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  respond(user_message, state)                               │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  SystemPrompter.build_system_prompt(                │    │
│  │      core={"coach_name": "Jaisu"},                  │    │
│  │      profile={"name": "Jan", "goal": "..."},        │    │
│  │      session={"phase": "discovery", "turn": 3},     │    │
│  │      history=[...]                                  │    │
│  │  )                                                  │    │
│  └─────────────────────────────────────────────────────┘    │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  main.j2  →  Renderuje system prompt                │    │
│  │                                                     │    │
│  │  "Jesteś Jasiu, coachem życiowym.                   │    │
│  │   Użytkownik ma na imię: Jan                        │    │
│  │   Cel: zmiana pracy                                 │    │
│  │   Faza: discovery, Tura: 3"                         │    │
│  └─────────────────────────────────────────────────────┘    │
│       │                                                     │
│       ▼                                                     │
│  call_llm(messages) → response                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

Jeżeli folder `life_coach_system/templates` nie istnieje, należy go utworzyć, uruchamiając komórkę poniżej.

In [ ]:
!mkdir -p life_coach_system/templates

## 🧠 Zadanie 5.5.6: Szablon promptu (`main.j2`)

W poprzednim zadaniu zdefiniowaliśmy klasę `SystemPrompter`, która szuka pliku `main.j2`. To najważniejszy plik tekstowy w całym projekcie. To tutaj instrukcje coachingowe spotykają się z Twoją architekturą danych.

Ponieważ używamy **Structured Output**, ten prompt pełni podwójną rolę:
1.  Musi instruować bota, jak ma się zachowywać (jako Coach).
2.  Musi instruować model, **jak poprawnie wypełnić pola** Twojego modelu Pydantic (`CoachResponseAnalysis`).

### Twoje zadanie:
Stwórz plik `life_coach_system/templates/main.j2`. Szablon musi być dynamiczny i reagować na stan rozmowy.

**Wymagania dla Twojego szablonu:**
*   **Obsługa faz:** Wykorzystaj przekazaną fazę, aby model wiedział, czy ma się witać, czy planować działania.
*   **Instrukcje dla pól:** Opisz modelowi, co ma wpisywać w `analysis_summary` (myśli), a co w `ai_response` (tekst dla usera).
*   **Dynamiczne powitanie:** Jeśli to pierwsza wiadomość (`turn_count == 0`), wymuś przedstawienie się z imienia (LC-001).
*   **Brak rad:** Wpisz "Prime Directive" (LC-003) w samym sercu promptu.

### 💡 Podpowiedź 1: Jakich danych możesz użyć?
Do Twojego szablonu trafiają cztery główne obiekty, które zdefiniowaliśmy w `CoachAgent`. Możesz się do nich odwoływać bezpośrednio:
*   `{{ core.coach_name }}` – Imię Twojego coacha z konfiguracji.
*   `{{ profile.user_name }}` – Imię użytkownika (jeśli już je znamy).
*   `{{ session.phase }}` – Aktualna faza rozmowy (np. `INTRODUCTION`, `EXPLORATION`).
*   `{{ session.turn_count }}` – Licznik wiadomości (kluczowy do zaliczenia LC-001).

### 💡 Podpowiedź 2: Składnia Jinja2 w praktyce
Pamiętaj, że w plikach `.j2` możesz używać logiki. Oto krótkie przypomnienie składni:
*   `{{ zmienna }}` – Wstawienie wartości.
*   `{% if warunek %} ... {% endif %}` – Logika warunkowa (np. sprawdzenie czy licznik tur wynosi 0).
*   `{# komentarz #}` – Komentarze, które nie trafią do finalnego promptu.

**Przykład logiki dla fazy:**
> *"Jesteś w fazie {{ session.phase }}. Twoim celem jest teraz {{ 'przywitanie się' if session.phase == 'INTRODUCTION' else 'zadawanie pytań' }}."*

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900; cursor: pointer;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć gotowy szablon 👈
    </summary>
<p>

```python
%%writefile life_coach_system/memory/templates/main.j2
{# =================================================================
   SYSTEM PROMPT DLA COACHA ZE STRUCTURED OUTPUT
   ================================================================= #}

Jesteś {{ core.coach_name }}, profesjonalnym coachem życiowym.

# TWOJA ROLA
Pomagasz użytkownikowi w rozwoju osobistym poprzez:
- Zadawanie pytań otwartych (Co? Jak? Dlaczego?)
- Aktywne słuchanie i parafrazowanie
- Rozpoznawanie emocji

# PRIME DIRECTIVE (LC-003)
ABSOLUTNIE ZAKAZANE jest dawanie gotowych rad!
Zamiast dawać rozwiązanie, pomóż użytkownikowi znaleźć własne.

# JAK WYPEŁNIĆ STRUKTURĘ ODPOWIEDZI (CoachResponseAnalysis)

## analysis_summary
To Twój wewnętrzny monolog. Zanim odpowiesz, przeanalizuj:
1. Co użytkownik właśnie powiedział i jakie emocje wykazuje?
2. Czy prosi o radę? (Jeśli tak, przygotuj się na asertywną odmowę).
3. Na jakim etapie jesteśmy? Czy już znamy jego imię i cel?

## coaching_phase
Wybierz fazę najlepiej opisującą obecny stan:
- INTRODUCTION (jeśli to początek)
- CONTEXT_GATHERING (jeśli zbierasz imię/cel)
- EXPLORATION (jeśli pogłębiasz problem)
- ACTION_PLANNING (jeśli planujecie konkretne kroki)

## ai_response
Finalna wiadomość. Musi być empatyczna, krótka i kończyć się pytaniem otwartym.

{# === DYNAMICZNY KONTEKST === #}

# UŻYTKOWNIK
{% if profile.user_name %}Imię: {{ profile.user_name }}{% else %}Imię: Nieznane (zapytaj o nie){% endif %}
{% if profile.main_goal %}Główny cel: {{ profile.main_goal }}{% endif %}

# SESJA
Aktualna faza: {{ session.phase | default('INTRODUCTION') }}
Numer tury: {{ session.turn_count | default(0) }}

{# === SPECJALNE INSTRUKCJE DLA STARTU (LC-001) === #}
{% if session.turn_count == 0 %}
W tej pierwszej wiadomości MUSISZ:
1. Przedstawić się jako {{ core.coach_name }}.
2. Wyjaśnić, że jesteś coachem strategicznym.
3. Zapytać użytkownika o imię.
{% endif %}
```

</p>
</details>





In [ ]:
%%writefile life_coach_system/templates/main.j2
{# =================================================================
   SYSTEM PROMPT DLA COACHA ZE STRUCTURED OUTPUT
   
   YOUR CODE HERE
   ================================================================= #}
    

Jesteś ...    

### Co to zmienia?
Zwróć uwagę na magię, która się tu dzieje. Dzięki Jinja2, jeśli `turn_count` wynosi 0, model dostaje **bardzo konkretny rozkaz**: "Przedstaw się". Gdy tylko licznik wzrośnie do 1, te instrukcje znikają z promptu, robiąc miejsce dla kolejnych faz. To właśnie dzięki temu system jest lekki, a model "nie gubi się" w nadmiarze niepotrzebnych w danym momencie poleceń. 

Już za moment, gdy masz już silnik, managera pamięci i prompt – czas uruchomić system i sprawdzić, jak radzi sobie na Leaderboardzie! Tylko jeszce jedna rzecz, trzeba trochę przepisać nasz `app.py`, aby teraz wprost korzystało z `coach.py` i innych rzeczy.


Struktura, do której dążymy:

```
life_coach_system/
│
├── app.py                          # 🎨 Gradio UI (główny punkt wejścia)
├── config.py                       # ⚙️  Konfiguracja centralna
│
├── engine/                         # 🧠 Logika biznesowa
│   ├── __init__.py
│   ├── coach.py                    # CoachAgent (GŁÓWNE TODO)
│   └── client.py                   # LLM client (Instructor)
│
├── memory/                         # 💾 Zarządzanie stanem
│   ├── __init__.py
│   ├── schemas/                    # Modele Pydantic
│   │   ├── __init__.py
│   │   └── session_state.py        # SessionState (TODO: rozszerzyć)
│   │
│   └── logic/                      # Logika pamięci
│       ├── __init__.py
│       └── manager.py              # MemoryManager (TODO: rozszerzyć)
│
├── persistence/                    # 💿 Zapis/odczyt danych
│   ├── __init__.py
│   ├── backend.py                  # ABC interface
│   └── in_memory.py                # Implementacja dev
│
├── utils/                          # 🔧 Narzędzia pomocnicze
│   ├── __init__.py
│   ├── evaluator.py                # LLM-as-Judge
│   └── leaderboard_parser.py       # Parser kryteriów
│
└── leaderboard_card_coach.md       # 📋 Kryteria oceny (LC-001 do LC-014)
```


### Diagram zależności 
Zanim zaczniesz uruchamiać komórki, spójrz jak te pliki ze sobą współpracują. To pomoże Ci nie zgubić się w strukturze:

```text
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           LIFE COACH SYSTEM - ARCHITEKTURA                          │
└─────────────────────────────────────────────────────────────────────────────────────┘

  [ INTERFEJS ]              [ LOGIKA BIZNESOWA ]              [ DANE / PAMIĘĆ ]
  ─────────────              ────────────────────              ─────────────────

  ┌──────────────┐           ┌────────────────────┐           ┌─────────────────────┐
  │   app.py     │◄─────────►│  engine/coach.py   │◄─────────►│ memory/logic/       │
  │  (Gradio UI) │           │   (CoachAgent)     │           │   manager.py        │
  └──────┬───────┘           └─────────┬──────────┘           │  (MemoryManager)    │
         │                             │                      └──────────┬──────────┘
         │                             │                                 │
         │                             ▼                                 ▼
         │                   ┌────────────────────┐           ┌─────────────────────┐
         │                   │ engine/client.py   │           │ memory/schemas/     │
         │                   │ (Instructor/LLM)   │           │  session_state.py   │
         │                   └─────────┬──────────┘           │  (Pydantic Models)  │
         │                             │                      └─────────────────────┘
         │                             ▼
         │                   ┌────────────────────┐           ┌─────────────────────┐
         │                   │    OpenAI API      │           │ persistence/        │
         │                   │                    │           │  backend.py (ABC)   │
         │                   └────────────────────┘           │  in_memory.py       │
         │                                                    └─────────────────────┘
         │
         ▼
  ┌──────────────┐           ┌────────────────────┐           ┌─────────────────────┐
  │   utils/     │◄──────────│ leaderboard_card_  │           │    config.py        │
  │ evaluator.py │           │    coach.md        │           │  (Config settings)  │
  │(LLM-as-Judge)│           └────────────────────┘           └─────────────────────┘
  └──────┬───────┘                    ▲
         │                            │
         ▼                   ┌────────┴───────────┐
  ┌──────────────┐           │ utils/leaderboard_ │
  │   Wyniki     │           │    parser.py       │
  │   oceny      │◄──────────│ (CheckDefinition)  │
  └──────────────┘           └────────────────────┘
```


## Persystencja: Gdzie trafiają dane?

Ostatnim elementem infrastruktury jest warstwa zapisu. Zaimplementujemy teraz dwa pliki:
*   `persistence/backend.py` – definicja abstrakcyjnego interfejsu.
*   `persistence/in_memory.py` – konkretna implementacja działająca w pamięci RAM.

To jest miejsce, w którym fizycznie "ląduje" pamięć Twojego systemu. W tej chwili, dla uproszczenia, trzymamy dane w pamięci operacyjnej, co oznacza, że znikną one po restarcie aplikacji. Jednak dzięki zastosowaniu tej architektury, możesz w przyszłości łatwo dopisać adaptery i przenosić stan sesji do baz takich jak Redis czy Postgres, nie zmieniając ani jednej linii kodu w samym agencie. Jest to najbardziej powtarzalna część systemu, którą będziesz mógł wykorzystywać w niemal każdym kolejnym projekcie.

## `life_coach_system/persistence/backend.py` 

In [ ]:
%%writefile life_coach_system/persistence/backend.py

"""
Persistence Backend Interface

Abstract interface for pluggable storage backends.
Implementations: InMemoryBackend (dev), RedisBackend (prod), PostgresBackend (future)

NOTE: Ten plik jest reu|ywalny - nie wymaga modyfikacji przez studenta.
"""

from abc import ABC, abstractmethod
from typing import Optional


class PersistenceBackend(ABC):
    """
    Abstract interface for user state persistence.

    All backends must implement these methods to be compatible with CoachAgent.
    """

    @abstractmethod
    def save(self, user_id: str, state: dict) -> None:
        """
        Save user state.

        Args:
            user_id: Unique user identifier
            state: Complete state dict (session_state z memory/schemas)

        Raises:
            PersistenceError: If save operation fails
        """
        pass

    @abstractmethod
    def load(self, user_id: str) -> Optional[dict]:
        """
        Load user state.

        Args:
            user_id: Unique user identifier

        Returns:
            State dict if user exists, None if new user

        Raises:
            PersistenceError: If load operation fails
        """
        pass

    @abstractmethod
    def exists(self, user_id: str) -> bool:
        """
        Check if user state exists.

        Args:
            user_id: Unique user identifier

        Returns:
            True if user has saved state, False otherwise
        """
        pass

    @abstractmethod
    def delete(self, user_id: str) -> None:
        """
        Delete user state.

        Args:
            user_id: Unique user identifier

        Raises:
            PersistenceError: If user doesn't exist
        """
        pass

    @abstractmethod
    def list_users(self) -> list[str]:
        """
        List all user IDs with saved state.

        Returns:
            List of user_id strings
        """
        pass


class PersistenceError(Exception):
    """Raised when persistence operations fail."""
    pass

## `life_coach_system/persistence/in_memory.py` 

In [ ]:
%%writefile life_coach_system/persistence/in_memory.py
"""
InMemory Persistence Backend

Simple in-memory storage for development and testing.
State is lost on app restart - use only for dev/testing.

NOTE: Ten plik jest reużywalny - nie wymaga modyfikacji przez studenta.
"""

from typing import Optional
from .backend import PersistenceBackend, PersistenceError
import copy


class InMemoryBackend(PersistenceBackend):
    """
    Store user state in memory (dict).

    Advantages:
    - Fast (no I/O)
    - Simple (no setup needed)
    - Good for development and testing

    Disadvantages:
    - State lost on restart
    - Not suitable for production
    - No multi-process support
    """

    def __init__(self):
        """Initialize empty storage."""
        self._storage: dict[str, dict] = {}

    def save(self, user_id: str, state: dict) -> None:
        """Save user state to memory."""
        # Deep copy to avoid reference issues
        self._storage[user_id] = copy.deepcopy(state)

    def load(self, user_id: str) -> Optional[dict]:
        """Load user state from memory."""
        state = self._storage.get(user_id)

        if state is None:
            return None

        # Return deep copy to avoid mutations affecting stored state
        return copy.deepcopy(state)

    def exists(self, user_id: str) -> bool:
        """Check if user has saved state."""
        return user_id in self._storage

    def delete(self, user_id: str) -> None:
        """Delete user state."""
        if not self.exists(user_id):
            raise PersistenceError(f"User {user_id} does not exist")

        del self._storage[user_id]

    def list_users(self) -> list[str]:
        """List all user IDs."""
        return list(self._storage.keys())

    def clear_all(self) -> None:
        """Clear all stored data (useful for testing)."""
        self._storage.clear()

    def get_storage_size(self) -> int:
        """Get number of users in storage."""
        return len(self._storage)


## `life_coach_system/app.py`

Czas na aktualizację pliku `app.py`. To nowa, pełna wersja interfejsu, która integruje wszystkie elementy: odświeżonego `CoachAgenta`, logikę pamięci oraz system persystencji.

Kod ten jest znacznie bardziej rozbudowany niż jego pierwsza wersja, ponieważ obsługuje teraz pełny cykl życia sesji i wizualizację ustrukturyzowanych danych. Nie musisz jednak analizować każdej linii tego pliku – potraktuj go jako gotową infrastrukturę UI. Twoja najważniejsza praca inżynierska odbywa się gdzie indziej: w **system prompcie** oraz w definicji **Structured Output**, ponieważ to one decydują o faktycznej inteligencji i jakości Twojego agenta.


In [ ]:
%%writefile life_coach_system/app.py

# -*- coding: utf-8 -*-
"""
Gradio UI for Life Coach System.

This file contains the complete user interface for the coaching system.
Run: python app.py
"""

import gradio as gr
import json
import tempfile
from datetime import datetime
from pathlib import Path
from engine.coach import CoachAgent
from persistence.in_memory import InMemoryBackend
from memory.logic.manager import MemoryManager
from memory.schemas.session_state import SessionState
from config import Config
from utils.leaderboard_parser import parse_leaderboard_card, filter_checks_by_priority
from utils.evaluator import evaluate_conversation as evaluate_conv_llm, format_evaluation_results

# ===================================================================
# Global instances (singleton pattern)
# ===================================================================

coach = CoachAgent()
storage = InMemoryBackend()
memory_manager = MemoryManager()


# ===================================================================
# UI handling functions
# ===================================================================

def interact(message: str, history: list, user_id: str):
    """
    Main function handling interaction with coach.

    Args:
        message: User message
        history: Chat history (Gradio messages format: list of dicts)
        user_id: User ID

    Returns:
        Tuple: (updated_history, state_dict, export_json)
    """
    if not message or not message.strip():
        return history, {"status": "No message"}, ""

    # Use default user_id if empty
    if not user_id or not user_id.strip():
        user_id = Config.DEFAULT_USER_ID

    # Get or create state
    if storage.exists(user_id):
        state_dict = storage.load(user_id)
        state = SessionState(**state_dict)
    else:
        state = memory_manager.create_empty_state(user_id)

    # Generate coach response
    try:
        response_text, updated_state = coach.respond(message, state)
    except Exception as e:
        error_msg = f"Error generating response: {str(e)}"
        print(f"ERROR: {error_msg}")
        return history, {"error": error_msg}, ""

    # Save state
    storage.save(user_id, updated_state.model_dump())

    # Update Gradio history (messages format: list of dicts)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response_text})

    # Prepare state visualization (dict for gr.JSON)
    state_dict = updated_state.model_dump()

    # Prepare export data (convert history to tuples for export)
    export_history = []
    for i in range(0, len(history) - 1, 2):
        if i + 1 < len(history):
            export_history.append((
                history[i].get("content", ""),
                history[i + 1].get("content", "")
            ))

    export_data = {
        "user_id": user_id,
        "exported_at": datetime.now().isoformat(),
        "conversation": export_history,
        "state": updated_state.model_dump()
    }
    export_json = json.dumps(export_data, indent=2, ensure_ascii=False)

    return history, state_dict, export_json


def reset_conversation(user_id: str):
    """
    Resets conversation (clears history and state).

    Args:
        user_id: User ID

    Returns:
        Tuple: (empty_history, message, empty_export)
    """
    if not user_id or not user_id.strip():
        user_id = Config.DEFAULT_USER_ID

    if storage.exists(user_id):
        storage.delete(user_id)

    return [], {"status": "State cleared"}, ""


def load_state_for_user(user_id: str):
    """
    Loads state for given user (to display when user_id changes).

    Args:
        user_id: User ID

    Returns:
        Tuple: (history, state_dict, export_json)
    """
    if not user_id or not user_id.strip():
        user_id = Config.DEFAULT_USER_ID

    if storage.exists(user_id):
        state_dict = storage.load(user_id)
        state = SessionState(**state_dict)

        # Reconstruct history for Gradio messages format (list of dicts)
        history = []
        for msg in state.conversation_history:
            history.append({
                "role": msg.get("role", "user"),
                "content": msg.get("content", "")
            })

        state_dict = state.model_dump()

        # Convert history to tuples for export
        export_history = []
        for i in range(0, len(history) - 1, 2):
            if i + 1 < len(history):
                export_history.append((
                    history[i].get("content", ""),
                    history[i + 1].get("content", "")
                ))

        export_data = {
            "user_id": user_id,
            "exported_at": datetime.now().isoformat(),
            "conversation": export_history,
            "state": state.model_dump()
        }
        export_json = json.dumps(export_data, indent=2, ensure_ascii=False)

        return history, state_dict, export_json
    else:
        return [], {"status": "No saved state for this user"}, ""


def export_to_file(export_json: str, user_id: str) -> str:
    """
    Exports conversation to temporary JSON file.

    Args:
        export_json: JSON string with conversation data
        user_id: User ID (for filename)

    Returns:
        str: Path to temporary file
    """
    if not export_json or export_json == "":
        return None

    # Create temp file with meaningful name
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"coach_conversation_{user_id}_{timestamp}.json"

    # Save to temp file
    temp_dir = Path(tempfile.gettempdir())
    filepath = temp_dir / filename

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(export_json)

    return str(filepath)


def evaluate_conversation(export_json: str, priority_filter: str, progress=gr.Progress()) -> str:
    """
    Evaluate conversation using LLM-as-Judge.

    Args:
        export_json: JSON string with conversation data
        priority_filter: "MUST-HAVE", "SHOULD-HAVE", or "ALL"
        progress: Gradio progress tracker

    Returns:
        Formatted evaluation results
    """
    progress(0, desc="Starting evaluation...")

    if not export_json or export_json == "":
        return "⚠️ **No conversation to evaluate.** Start a conversation first."

    # Parse export data
    progress(0.1, desc="Parsing conversation...")
    try:
        export_data = json.loads(export_json)
        conversation = export_data.get("state", {}).get("conversation_history", [])

        if not conversation:
            return "⚠️ **No messages in conversation.** Have a conversation with the coach first."

    except Exception as e:
        return f"❌ **Error parsing conversation:** {str(e)}"

    # Parse leaderboard card
    progress(0.2, desc="Loading evaluation criteria...")
    try:
        leaderboard_path = Path(__file__).parent / "leaderboard_card_coach.md"
        all_checks = parse_leaderboard_card(str(leaderboard_path))

        if not all_checks:
            return "❌ **Error:** No criteria found in leaderboard card."

    except Exception as e:
        return f"❌ **Error parsing leaderboard:** {str(e)}"

    # Filter checks by priority
    progress(0.3, desc=f"Filtering criteria ({priority_filter})...")
    filtered_checks = filter_checks_by_priority(all_checks, priority_filter)

    if not filtered_checks:
        return f"⚠️ **No criteria match filter:** {priority_filter}"

    # Evaluate using LLM
    progress(0.4, desc=f"Evaluating {len(filtered_checks)} criteria with LLM...")
    try:
        eval_result = evaluate_conv_llm(conversation, filtered_checks)

        if "error" in eval_result:
            return f"❌ **Evaluation error:** {eval_result['error']}"

        # Format results
        progress(0.9, desc="Formatting results...")
        formatted = format_evaluation_results(eval_result)

        progress(1.0, desc="Done!")
        return formatted

    except Exception as e:
        import traceback
        error_detail = traceback.format_exc()
        return f"❌ **Evaluation failed:**\n\n```\n{error_detail}\n```"


# ===================================================================
# Building Gradio interface
# ===================================================================

with gr.Blocks(
    title="Life Coach System",
    theme=gr.themes.Default()
) as demo:
    gr.Markdown("# Life Coach System")
    gr.Markdown("**Features:** Multi-user • Memory State • Leaderboard Evaluation")

    with gr.Row():
        # ===============================================================
        # LEFT COLUMN: Chat Interface (70%)
        # ===============================================================
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(
                type="messages",
                height=600,
                show_label=False
            )

            msg = gr.Textbox(
                placeholder="Message...",
                lines=2,
                show_label=False
            )

            with gr.Row():
                send_btn = gr.Button("Send", variant="primary", scale=3)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        # ===============================================================
        # RIGHT COLUMN: Sidebar (30%)
        # ===============================================================
        with gr.Column(scale=3, variant="panel"):
            gr.Markdown("### Controls")

            user_id_input = gr.Textbox(
                label="User ID",
                value=Config.DEFAULT_USER_ID,
                placeholder="Enter your ID",
                info="Each user has independent state"
            )

            # Memory State (collapsed)
            with gr.Accordion("🧠 Memory State", open=False):
                state_viewer = gr.JSON(
                    label="Session State",
                    value={"status": "No state - start conversation"}
                )

            # Export (collapsed)
            with gr.Accordion("⚙️ Advanced", open=False):
                gr.Markdown("**Export Session**")

                export_data = gr.Textbox(
                    label="Export data (JSON)",
                    lines=1,
                    visible=False  # Hidden field for storing JSON
                )

                export_btn = gr.Button("📥 Export JSON", size="sm")

                export_file = gr.File(
                    label="Download",
                    visible=True
                )

            # Leaderboard (collapsed) - NEW
            with gr.Accordion("📊 Leaderboard", open=False):
                gr.Markdown("**Evaluate Conversation Quality**")

                priority_dropdown = gr.Dropdown(
                    choices=["ALL", "MUST-HAVE", "SHOULD-HAVE"],
                    value="MUST-HAVE",
                    label="Priority Filter",
                    info="Select which criteria to evaluate",
                    interactive=True
                )

                evaluate_btn = gr.Button("🎯 Evaluate", variant="primary", size="sm")

                leaderboard_results = gr.Markdown(
                    value="*Click 'Evaluate' to see results...*"
                )


    # ===================================================================
    # Event handlers
    # ===================================================================

    # Send message (button)
    send_btn.click(
        fn=interact,
        inputs=[msg, chatbot, user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    ).then(
        lambda: "",  # Clear text field after sending
        None,
        msg
    )

    # Send message (Enter)
    msg.submit(
        fn=interact,
        inputs=[msg, chatbot, user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    ).then(
        lambda: "",  # Clear text field
        None,
        msg
    )

    # Reset conversation
    clear_btn.click(
        fn=reset_conversation,
        inputs=[user_id_input],
        outputs=[chatbot, state_viewer, export_data]
    )

    # Export conversation to file
    export_btn.click(
        fn=export_to_file,
        inputs=[export_data, user_id_input],
        outputs=[export_file]
    )

    # Evaluate conversation
    evaluate_btn.click(
        fn=evaluate_conversation,
        inputs=[export_data, priority_dropdown],
        outputs=[leaderboard_results],
        show_progress="full"  # Show progress bar and disable button during execution
    )


# ===================================================================
# Launch application
# ===================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("Life Coach System - MVP")
    print("=" * 70)
    print(f"Model: {Config.MODEL_NAME}")
    print(f"Coach Name: {Config.COACH_NAME}")
    print(f"Storage: {Config.PERSISTENCE_BACKEND}")
    print(f"Debug Mode: {Config.DEBUG}")
    print("=" * 70)

    demo.launch(
        server_name="0.0.0.0",
        server_port=8080,
        share=False  # Set True if you want public link
    )



In [ ]:
!python life_coach_system/app.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈


Przypominam, że Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej.
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera.
3.  Dopiero gdy `[*]` zniknie, przejdź dalej.

Jeśli poprawnie zdefiniowane:
- **Structured Output**
- **System Prompt** (`main.j2`), powinieneś zobaczyć wynik, na który pracowałeś przez cały kurs: **wszystkie 7 testów MUST-HAVE zaliczonych na zielono**.

To oficjalny dowód na to, że Twój system spełnia założenia projektowe i kwalifikuje się do uzyskania certyfikatu końcowego.

![](../images/ai_coach_pass.jpg)

Oczywiście, w ramach tej lekcji zastosowaliśmy pewne inżynierskie uproszczenia, aby sprawnie przejść przez proces budowy. Moim celem było pokazanie Ci konkretnych kierunków i obszarów zarządzania systemem agentowym, a nie dostarczenie zamkniętego, sztywnego pudełka.

Wierzę, że ten projekt stanie się Twoim nowym punktem odniesienia (**baseline**). Masz teraz solidną architekturę, sprawne zarządzanie pamięcią i – co najważniejsze – mechanizm automatycznej ewaluacji, który nie pozwoli Ci błądzić po omacku.

Właśnie ukończyłeś budowę kompletnego systemu w paradygmacie Software 3.0. Dalej możesz rozwijać go w dowolnym kierunku: dodawać nowe umiejętności (Skills), pogłębiać analizę emocji czy rozbudowywać fazy coachingu. Od tego momentu – jak to się mówi – *sky is the limit*.

**Gratulacje!**

## Podsumowanie

Gratulacje! To był intensywny proces, który właśnie dobiegł końca. Udało się domknąć cykl tworzenia zaawansowanego systemu Agentic AI. Patrząc wstecz na Moduł 5 i cały program, widać jak ogromną drogę udało się przejść.

Zaczynając od Lekcji 1.1, rozmawialiśmy o odpowiedzialności inżynierskiej i braniu pełnej kontroli nad systemem. Wtedy Spektrum Autonomii AI było tylko teorią na slajdach. Dzisiaj to Spektrum jest Twoim narzędziem pracy. 
![](../../module1/images/software3.0_7_levels.jpg)

Przejście przez poziomy 4 (Contextual AI – pamięć i narzędzia) oraz 5 (Strategic AI – adaptacyjne planowanie) stało się faktem. To jest coś znacznie więcej niż tylko "hype", to architektura, którą potrafisz zaimplementować.

Kluczowym wnioskiem z całego kursu jest zrozumienie, czym naprawdę jest **Context Engineering**. To nie jest tylko pisanie długich instrukcji. To precyzyjne składanie wielu źródeł danych w jedną, spójną całość, którą LLM zrozumie bezbłędnie:
- **Warstwy systemowe**: Dynamicznie wstrzykiwane prompty zależne od fazy rozmowy.
- **Warstwy pamięci**: Dane semantyczne (profil użytkownika) i epizodyczne (historia).
- **Warstwy narzędziowe**: Możliwości interakcji ze światem zewnętrznym.
- **Warstwy decyzyjne**: Adaptacyjne plany, które zmieniają się, gdy sytuacja tego wymaga.

Poniższy diagram ilustruje tę inżynierską sztukę składania kontekstu:

![](../../module1/images/context_engineering.jpg)

Teraz widać to wyraźnie: finalnie wszystko sprowadza się do umiejętności poskładania tych klocków w taki systemowy prompt, który model "łyknie" z maksymalną precyzją. Co ciekawe, ta technika jest niezwykle odporna na przyszłe zmiany modeli. Nawet słabsze (ale dobre) modele działają świetnie, jeśli dostaną krótki, precyzyjny i dobrze wyrzeźbiony kontekst. 

Ludzi na LinkedIn często podpowiadają nam, że "więcej znaczy lepiej". W AI to błąd! Wrzucanie 200 tysięcy tokenów i tym bardziej milliony do "dużego worka" (Long Context) zwykle kończy się źle – zarówno pod względem kosztów, czasu wykonania, jak i – co najgorsze – jakości. Prawdziwe rzemiosło to selekcja danych, a nie ich składowanie.

Dzięki temu programowi zyskujesz fundamenty, które nie zestarzeją się wraz z kolejną wersją GPT czy Claude’a. Masz w ręku proces budowania systemów stateful dookoła statelessowych modeli. 

Dobra wiadomość jest taka: to dopiero początek Twojej drogi w świecie Software 3.0. Budowanie systemów, które uczą się z każdej interakcji i działają w realnym świecie, to dzisiaj najbardziej poszukiwana kompetencja inżynierska.

**Wow 😮. To była niesamowita podróż. Do zobaczenia na produkcji!**